In [1]:
# ============================================================================
# CELL 1: Thread-pool caps
# ============================================================================
import os

_N_THREADS = 4

os.environ["OMP_NUM_THREADS"]        = str(_N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"]   = str(_N_THREADS)
os.environ["MKL_NUM_THREADS"]        = str(_N_THREADS)
os.environ["NUMEXPR_NUM_THREADS"]    = str(_N_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(_N_THREADS)
os.environ["BLIS_NUM_THREADS"]       = str(_N_THREADS)
os.environ["OPENBLAS_NUM_PARALLEL"]  = "1"

In [2]:
# ============================================================================
# CELL 2: Imports
# ============================================================================

# --- Project Package (installed via `pip install -e .`) ---

# --- Project Package ---
#from clinical_trial_benchmarking.clustering import (
#    ClusterPreprocessor,
#    ClusterPredictor,
#)
from clinical_trial_benchmarking.utils import safe_parse_list
from clinical_trial_benchmarking.taxonomy import (
    SPONSOR_DISPLAY,
    TA_DISPLAY,
    TA_STRING_TO_FLAG,
    THERAPEUTIC_AREA_COLUMNS,
    THERAPEUTIC_AREA_KEYWORDS,
    OBSERVATIONAL_MODEL_VALUES,
    OBSERVATIONAL_PERSPECTIVE_VALUES,
    OBSERVATIONAL_MODEL_MAP,
    OBSERVATIONAL_PERSPECTIVE_MAP,
)

# --- Core Utilities ---
import ast
import gc
import json
import os
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path

# --- Data Handling ---
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing ---
from sklearn.preprocessing import RobustScaler

# --- Attribution ---
import shap

# --- Regression model for attribution (not deployment) ---
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# --- Misc ---
from scipy.stats import kruskal

warnings.filterwarnings("ignore")

In [3]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================

import pyarrow.parquet as pq

_PARQUET_PATH = 'clinical_trials_major_diseases.parquet'

_RAW_COLS_NEEDED = [
    'protocolSection.identificationModule.nctId',
    'protocolSection.designModule.studyType',
    'protocolSection.designModule.phases',
    'protocolSection.designModule.enrollmentInfo.count',
    'protocolSection.designModule.targetDuration',
    'protocolSection.designModule.designInfo.primaryPurpose',
    'protocolSection.designModule.designInfo.maskingInfo.masking',
    'protocolSection.designModule.designInfo.timePerspective',
    'protocolSection.designModule.designInfo.observationalModel',
    'protocolSection.armsInterventionsModule.armGroups',
    'protocolSection.eligibilityModule.eligibilityCriteria',
    'protocolSection.eligibilityModule.minimumAge',
    'protocolSection.eligibilityModule.maximumAge',
    'protocolSection.conditionsModule.conditions',
    'protocolSection.contactsLocationsModule.locations',
    'protocolSection.outcomesModule.primaryOutcomes',
    'protocolSection.outcomesModule.secondaryOutcomes',
    'protocolSection.outcomesModule.otherOutcomes',
    'protocolSection.identificationModule.organization.class',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.class',
    'protocolSection.statusModule.startDateStruct.date',
    'protocolSection.statusModule.completionDateStruct.date',
    'hasResults',
]

_schema = pq.read_schema(_PARQUET_PATH)
_available = set(_schema.names)
_missing = [c for c in _RAW_COLS_NEEDED if c not in _available]
if _missing:
    raise RuntimeError(
        "❌ Missing columns in source parquet:\n"
        + "\n".join(f"   - {c}" for c in _missing)
    )

print("=" * 78)
print("LOADING DATA (projected read)")
print("=" * 78)
print(f"  Source:  {_PARQUET_PATH}")
print(f"  Cols in file: {len(_available)}")
print(f"  Cols read:    {len(_RAW_COLS_NEEDED)}")

df = pd.read_parquet(_PARQUET_PATH, columns=_RAW_COLS_NEEDED)
print(f"  ✅ Loaded {len(df):,} rows × {df.shape[1]} columns")
print("=" * 78)

LOADING DATA (projected read)
  Source:  clinical_trials_major_diseases.parquet
  Cols in file: 148
  Cols read:    23
  ✅ Loaded 541,644 rows × 23 columns


In [4]:
# ============================================================================
# CELL 3.1: Normalize array-like cells in object columns
# ============================================================================

from collections import Counter

_ARRAY_LIKE_TYPES = (np.ndarray,)
try:
    import pyarrow as pa
    _ARRAY_LIKE_TYPES += (pa.Array, pa.ChunkedArray, pa.ListArray, pa.LargeListArray)
except ImportError:
    pa = None


def _is_arraylike(x) -> bool:
    if x is None:
        return False
    if isinstance(x, _ARRAY_LIKE_TYPES):
        return True
    if hasattr(x, '__array__') and not isinstance(x, (str, bytes, dict)):
        return True
    if hasattr(x, 'to_pylist') and callable(x.to_pylist):
        return True
    return False


class _UnwrapError(Exception):
    pass


def _to_pylist(x, _depth=0, _path="<root>"):
    if _depth > 25:
        raise _UnwrapError(f"max depth exceeded at {_path}")
    if x is None:
        return x
    if isinstance(x, np.ndarray):
        return [_to_pylist(v, _depth + 1, f"{_path}[{i}]")
                for i, v in enumerate(x.tolist())]
    if pa is not None and isinstance(x, _ARRAY_LIKE_TYPES[1:]):
        try:
            return [_to_pylist(v, _depth + 1, f"{_path}[{i}]")
                    for i, v in enumerate(x.to_pylist())]
        except Exception as e:
            raise _UnwrapError(f"Arrow to_pylist failed at {_path}: {e}")
    if hasattr(x, '__array__') and not isinstance(x, (str, bytes, dict)):
        try:
            return [_to_pylist(v, _depth + 1, f"{_path}[{i}]")
                    for i, v in enumerate(np.asarray(x).tolist())]
        except Exception as e:
            raise _UnwrapError(f"__array__ failed at {_path}: {e}")
    if hasattr(x, 'to_pylist') and callable(x.to_pylist):
        try:
            return [_to_pylist(v, _depth + 1, f"{_path}[{i}]")
                    for i, v in enumerate(x.to_pylist())]
        except Exception as e:
            raise _UnwrapError(f"to_pylist failed at {_path}: {e}")
    if isinstance(x, (list, tuple)):
        return [_to_pylist(v, _depth + 1, f"{_path}[{i}]")
                for i, v in enumerate(x)]
    if isinstance(x, dict):
        return {k: _to_pylist(v, _depth + 1, f"{_path}.{k}")
                for k, v in x.items()}
    return x


def _column_has_arraylike(series: pd.Series) -> bool:
    if series.dtype != object:
        return False
    nonnull = series.dropna()
    if len(nonnull) == 0:
        return False
    try:
        n_unique = nonnull.nunique(dropna=True)
    except TypeError:
        return nonnull.apply(_is_arraylike).any()
    if n_unique <= 5000:
        return any(_is_arraylike(v) for v in nonnull.unique())
    idx = np.linspace(0, len(nonnull) - 1, min(10_000, len(nonnull))).astype(int)
    return any(_is_arraylike(v) for v in nonnull.iloc[idx])


def _normalize_object_columns(df_in):
    provenance = {}
    n_converted = 0
    n_failed = 0
    for col in df_in.columns:
        if df_in[col].dtype != object:
            continue
        if not _column_has_arraylike(df_in[col]):
            provenance[col] = 'clean'
            continue
        _failed_row = None
        _failed_msg = None

        def _safe_wrap(v, path, _col=col):
            nonlocal _failed_row, _failed_msg
            try:
                return _to_pylist(v, _path=path)
            except _UnwrapError as e:
                if _failed_row is None:
                    _failed_row = path
                    _failed_msg = str(e)
                return v

        try:
            df_in[col] = [_safe_wrap(v, f"{col}[{i}]")
                          for i, v in enumerate(df_in[col].values)]
        except Exception as e:
            provenance[col] = f'failed: {e}'
            n_failed += 1
            continue
        if _failed_row is not None:
            provenance[col] = f'failed:{_failed_msg}'
            n_failed += 1
        else:
            provenance[col] = 'converted'
            n_converted += 1
    return df_in, provenance, n_converted, n_failed


print("\n" + "=" * 70)
print("NORMALIZE OBJECT COLUMNS")
print("=" * 70)
df, _provenance, _n_conv, _n_fail = _normalize_object_columns(df)
print(f"  Converted: {_n_conv}  Failed: {_n_fail}")

_residual = []
for _col in df.columns:
    if df[_col].dtype != object:
        continue
    try:
        if df[_col].apply(_is_arraylike).any():
            _residual.append(_col)
    except Exception as e:
        _residual.append(f"{_col} ({e})")

if _residual:
    raise RuntimeError(
        f"❌ {len(_residual)} object column(s) still contain array-likes:\n"
        + "\n".join(f"   - {c}" for c in _residual)
    )

del _provenance, _residual
gc.collect()
print("=" * 70)


NORMALIZE OBJECT COLUMNS
  Converted: 7  Failed: 0


In [5]:
# ============================================================================
# CELL 3.2: Canonical split + directory tree
# ============================================================================
# NOTE: In this notebook, `_split` is NOT a train/test split for model
# evaluation. It is a holdout fold used ONLY for:
#   - fitting the global attribution model (train fold)
#   - stability diagnostics (does peer-group membership drift across folds?)
# The audit itself reports over the FULL corpus — every trial is a peer.
#
# The split is stratified by study type, and remains canonical (same seed,
# same size) so the downstream notebook sees a stable contract.

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

_SHARED_ROOT   = Path('cluster_data/shared')
_STRATEGY_ROOT = Path('cluster_data/strategies')  # kept for compat

_SHARED_ROOT.mkdir(parents=True, exist_ok=True)

_stratify_col = 'protocolSection.designModule.studyType'

if '_split' in df.columns and set(df['_split'].dropna().unique()) == {'train', 'test'}:
    print("  ℹ️ '_split' already exists — reusing.")
else:
    df['_split'] = 'train'
    _train_idx, _test_idx = train_test_split(
        df.index, test_size=0.2, random_state=42,
        stratify=df[_stratify_col],
    )
    df.loc[_test_idx, '_split'] = 'test'

_n_train = int((df['_split'] == 'train').sum())
_n_test  = int((df['_split'] == 'test').sum())
print("=" * 70)
print("CANONICAL SPLIT (attribution-only; audit runs over full corpus)")
print("=" * 70)
print(f"  Train (attribution): {_n_train:,}   Test (stability): {_n_test:,}   Total: {len(df):,}")
print("=" * 70)

gc.collect();

CANONICAL SPLIT (attribution-only; audit runs over full corpus)
  Train (attribution): 433,315   Test (stability): 108,329   Total: 541,644


In [6]:
# ============================================================================
# CELL 3.3: Helper functions
# ============================================================================

def count_items(items):
    if isinstance(items, list):
        return len(items)
    return 0


def count_sites(locations):
    if isinstance(locations, np.ndarray):
        return int(locations.size)
    if hasattr(locations, 'to_pylist'):
        try:
            locations = locations.to_pylist()
        except Exception:
            return 0
    if isinstance(locations, (list, tuple)):
        return len(locations)
    if locations is None:
        return 0
    try:
        if pd.isna(locations):
            return 0
    except (ValueError, TypeError):
        return 0
    if isinstance(locations, str):
        s = locations.strip()
        if s in ('', '[]', 'nan', 'None', 'null', 'NULL'):
            return 0
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
    return 0


def extract_age_simple(age_str):
    if pd.isna(age_str):
        return 0
    age_str = str(age_str).strip()
    numbers = ''.join(c for c in age_str if c.isdigit() or c == '.')
    if numbers and numbers != '.':
        return float(numbers)
    m = re.search(r'(\d+\.?\d*)', age_str)
    return float(m.group(1)) if m else 0


def get_study_type(type_val):
    return 'UNKNOWN' if pd.isna(type_val) else str(type_val)


def get_phase_indicator(phase, phase_name):
    if pd.isna(phase):
        return 0
    if isinstance(phase, list):
        phase = str(phase)
    return 1 if phase_name in str(phase).upper() else 0


def count_criteria_words(criteria):
    if pd.isna(criteria):
        return 0
    return len(str(criteria).split())


def count_countries(locations):
    if not isinstance(locations, list):
        return 0
    countries = set()
    for loc in locations:
        if isinstance(loc, dict) and 'country' in loc:
            countries.add(loc['country'])
    return len(countries)


def count_us_sites(locations):
    if not isinstance(locations, list):
        return 0
    return sum(1 for loc in locations
               if isinstance(loc, dict) and loc.get('country') == 'United States')


def extract_years(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if s in ('', 'nan', 'None', '0', '0.0'):
        return np.nan
    m = re.search(r'(\d+\.?\d*)', s)
    if not m:
        return np.nan
    n = float(m.group(1))
    if n == 0:
        return np.nan
    t = s.lower()
    if 'month' in t:  return n / 12
    if 'week' in t:   return n / 52.1775
    if 'day' in t:    return n / 365.25
    return n


_TA_KEYWORDS = {
    f'is_{name}': keys
    for name, keys in THERAPEUTIC_AREA_KEYWORDS.items()
}

In [7]:
# ============================================================================
# CELL 3.4: FIXED OUTCOME COMPLEXITY FEATURES + RAW LOCATIONS DIAGNOSTIC
# ============================================================================

# ---------------------------------------------------------------------------
# 3.1a  RAW LOCATIONS COLUMN DIAGNOSTIC
# ---------------------------------------------------------------------------
_LOC_COL = 'protocolSection.contactsLocationsModule.locations'

if _LOC_COL not in df.columns:
    print(f"⚠️  Column '{_LOC_COL}' not found — skipping raw locations diagnostic.")
else:
    _raw = df[_LOC_COL]

    n_total = len(_raw)
    n_nan   = _raw.isna().sum()

    # Per-row counts — never materialize a full string column.
    def _is_empty_like(v):
        if isinstance(v, (list, tuple, np.ndarray)):
            return len(v) == 0
        if hasattr(v, 'to_pylist'):
            try:
                return len(v.to_pylist()) == 0
            except Exception:
                return False
        if v is None:
            return True
        try:
            if pd.isna(v):
                return True
        except (ValueError, TypeError):
            return False
        s = str(v).strip()
        return s in ('', '[]', 'nan', 'None', 'null', 'NULL')

    n_empty_list = int(_raw.apply(_is_empty_like).sum())
    n_null_str   = int(
        _raw.apply(
            lambda v: isinstance(v, str)
            and v.strip() in ('nan', 'None', 'null', 'NULL')
        ).sum()
    )

    def _parse_status(val):
        if isinstance(val, np.ndarray):
            return 'already_list' if val.size > 0 else 'empty'
        if hasattr(val, 'to_pylist'):
            try:
                _pl = val.to_pylist()
                return 'already_list' if len(_pl) > 0 else 'empty'
            except Exception:
                return 'unparseable'
        if isinstance(val, (list, tuple)):
            return 'already_list' if len(val) > 0 else 'empty'

        if val is None:
            return 'nan'
        try:
            if pd.isna(val):
                return 'nan'
        except (ValueError, TypeError):
            return 'already_list'

        s = str(val).strip()
        if s in ('', '[]', 'nan', 'None', 'null', 'NULL'):
            return 'empty'
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return 'json_list' if len(parsed) > 0 else 'empty'
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return 'ast_list' if len(parsed) > 0 else 'empty'
        except Exception:
            pass
        return 'unparseable'

    _status = _raw.apply(_parse_status)
    _status_counts = _status.value_counts()

    print("\n" + "=" * 70)
    print("RAW LOCATIONS COLUMN DIAGNOSTIC")
    print("=" * 70)
    print(f"  Total rows:              {n_total:,}")
    print(f"  NaN:                     {n_nan:,}  ({n_nan/n_total*100:5.1f}%)")
    print(f"  Empty list '[]':         {n_empty_list:,}  ({n_empty_list/n_total*100:5.1f}%)")
    print(f"  Null-like strings:       {n_null_str:,}  ({n_null_str/n_total*100:5.1f}%)")
    print(f"\n  Parse-status distribution:")
    for status, count in _status_counts.items():
        print(f"    {status:<14}  {count:>8,}  ({count/n_total*100:5.1f}%)")

    _unparseable = _status_counts.get('unparseable', 0)
    if _unparseable > 0:
        print(f"\n  ⚠️  {_unparseable:,} rows FAILED to parse as a list.")
        print("     These will silently become num_sites == 0 downstream.")
        print("     Inspect a sample before trusting `num_sites`:")
        _bad_sample = _raw[_status == 'unparseable'].head(3)
        for i, v in enumerate(_bad_sample, 1):
            _preview = str(v)[:120].replace('\n', ' ')
            print(f"       [{i}] {_preview!r}")
    else:
        print("\n  ✅ No unparseable rows — zero-counts are genuine empties.")

    if 'protocolSection.designModule.studyType' in df.columns:
        _st = df['protocolSection.designModule.studyType'].fillna('UNKNOWN')
        _ct = pd.crosstab(_st, _status, normalize='index') * 100
        print("\n  Parse-status % by study type:")
        print(_ct.round(1).to_string())
    else:
        print("\n  (studyType column not found — skipping per-type breakdown.)")

    print("=" * 70)

    # Release the transient diagnostic structures before the outcome-parsing
    # block below, which allocates its own large intermediates.
    del _status, _status_counts, _raw
    gc.collect()


# ---------------------------------------------------------------------------
# 3.1b  Outcome parsing helpers
# ---------------------------------------------------------------------------

def parse_outcome_data(outcome_data):
    """Parse outcome data whether it's a list or string representation"""
    if isinstance(outcome_data, np.ndarray):
        return outcome_data.tolist()
    if hasattr(outcome_data, 'to_pylist'):
        try:
            return outcome_data.to_pylist()
        except Exception:
            return []
    if isinstance(outcome_data, (list, tuple)):
        return list(outcome_data)

    if outcome_data is None:
        return []
    try:
        if pd.isna(outcome_data):
            return []
    except (ValueError, TypeError):
        return []

    if isinstance(outcome_data, str):
        try:
            parsed = ast.literal_eval(outcome_data)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

        try:
            matches = re.findall(r'\{[^}]*\}', outcome_data)
            if matches:
                result = []
                for m in matches:
                    try:
                        dict_str = m.replace("'", '"')
                        result.append(json.loads(dict_str))
                    except Exception:
                        pass
                if result:
                    return result
        except Exception:
            pass

    return []


def count_outcomes(outcome_data):
    """Count outcomes from parsed data"""
    parsed = parse_outcome_data(outcome_data)
    return len(parsed)


def get_outcome_types_from_data(outcome_data):
    """Extract outcome types from parsed data"""
    parsed = parse_outcome_data(outcome_data)
    outcome_types = set()

    for outcome in parsed:
        if isinstance(outcome, dict):
            outcome_str = str(outcome).upper()
        elif isinstance(outcome, str):
            outcome_str = outcome.upper()
        else:
            continue

        if any(key in outcome_str for key in ['EFFICACY', 'EFFECTIVENESS']):
            outcome_types.add('EFFICACY')
        if any(key in outcome_str for key in ['SAFETY', 'ADVERSE']):
            outcome_types.add('SAFETY')
        if any(key in outcome_str for key in ['QUALITY', 'QOL']):
            outcome_types.add('QUALITY_OF_LIFE')
        if any(key in outcome_str for key in ['SURVIVAL', 'MORTALITY']):
            outcome_types.add('SURVIVAL')
        if 'BIOMARKER' in outcome_str:
            outcome_types.add('BIOMARKER')
        if 'PHARMACOKINETIC' in outcome_str or 'PK' in outcome_str:
            outcome_types.add('PHARMACOKINETIC')
        if 'PHARMACODYNAMIC' in outcome_str or 'PD' in outcome_str:
            outcome_types.add('PHARMACODYNAMIC')
        if any(key in outcome_str for key in ['PATIENT', 'REPORTED']):
            outcome_types.add('PATIENT_REPORTED')

    return outcome_types


def get_outcome_complexity(outcome_data):
    """Calculate complexity score based on outcome data"""
    parsed = parse_outcome_data(outcome_data)
    if not parsed:
        return 0

    complexity = 0
    for outcome in parsed:
        if isinstance(outcome, dict):
            outcome_str = str(outcome).upper()
            if any(key in outcome_str for key in ['EFFICACY', 'EFFECTIVENESS']):
                complexity += 1
            if any(key in outcome_str for key in ['SAFETY', 'ADVERSE']):
                complexity += 1
            if any(key in outcome_str for key in ['QUALITY', 'QOL']):
                complexity += 1
            if any(key in outcome_str for key in ['SURVIVAL', 'MORTALITY']):
                complexity += 1
            if 'BIOMARKER' in outcome_str:
                complexity += 1
            if 'PHARMACOKINETIC' in outcome_str or 'PK' in outcome_str:
                complexity += 1
            if 'PHARMACODYNAMIC' in outcome_str or 'PD' in outcome_str:
                complexity += 1

    return min(complexity, 10)


# ---------------------------------------------------------------------------
# 3.1c  Apply to columns
# ---------------------------------------------------------------------------

print("\nParsing outcome data...")

df['num_primary_outcomes'] = df['protocolSection.outcomesModule.primaryOutcomes'].apply(count_outcomes)
df['num_secondary_outcomes'] = df['protocolSection.outcomesModule.secondaryOutcomes'].apply(count_outcomes)
df['num_other_outcomes'] = df['protocolSection.outcomesModule.otherOutcomes'].apply(count_outcomes)
df['total_outcomes'] = df['num_primary_outcomes'] + df['num_secondary_outcomes'] + df['num_other_outcomes']

df['outcome_types'] = df['protocolSection.outcomesModule.primaryOutcomes'].apply(get_outcome_types_from_data)

df['outcome_complexity_total'] = df['protocolSection.outcomesModule.primaryOutcomes'].apply(get_outcome_complexity) + \
                                  df['protocolSection.outcomesModule.secondaryOutcomes'].apply(get_outcome_complexity)

outcome_types_list = ['EFFICACY', 'SAFETY', 'QUALITY_OF_LIFE', 'SURVIVAL',
                      'BIOMARKER', 'PHARMACOKINETIC', 'PHARMACODYNAMIC', 'PATIENT_REPORTED']

for ot in outcome_types_list:
    df[f'has_{ot.lower()}_outcome'] = df['outcome_types'].apply(lambda x: ot in x)

df['outcome_type_count'] = df['outcome_types'].apply(len)
df['outcome_type_diversity'] = df['outcome_types'].apply(lambda x: len(x) / 8 if x else 0)

# ---------------------------------------------------------------------------
# 3.1d  Verification (DIAGNOSTIC ONLY — NOT CONTRACTUAL)
# ---------------------------------------------------------------------------
# The numbers below are a runtime sanity check on the parsing logic above,
# not a specification. They describe THIS run's input parquet and will
# change whenever:
#
#   - the source parquet is regenerated (notebook 01 is re-run)
#   - the CT.gov query in notebook 01 is extended or narrowed
#   - an upstream filter changes the row universe before this cell runs
#
# Do NOT gate downstream logic on these values, and do NOT treat a drift in
# any of them as a pipeline failure on its own. The load-bearing contracts
# for outcome features are:
#
#   - the column names produced here (num_primary_outcomes, total_outcomes,
#     has_efficacy_outcome, outcome_type_count, outcome_type_diversity, ...)
#   - the dtypes of those columns (all int or bool)
#   - the invariant that total_outcomes == num_primary + num_secondary +
#     num_other
#
# Those are asserted by Cell 15, not by the prints below. If you need a
# stable regression check on outcome-feature distributions, snapshot the
# prints to a file and diff them manually between runs.
# ---------------------------------------------------------------------------

print("\n=== VERIFICATION (diagnostic; not contractual) ===")
print(f"Primary outcomes - min: {df['num_primary_outcomes'].min()}, "
      f"max: {df['num_primary_outcomes'].max()}, "
      f"mean: {df['num_primary_outcomes'].mean():.2f}")
print(f"Total outcomes - min: {df['total_outcomes'].min()}, "
      f"max: {df['total_outcomes'].max()}, "
      f"mean: {df['total_outcomes'].mean():.2f}")
print(f"Trials with outcomes: {(df['total_outcomes'] > 0).sum()} "
      f"({(df['total_outcomes'] > 0).sum() / len(df) * 100:.1f}%)")

# Invariant that IS contractual — asserted here as well as in Cell 15.
assert (
    df['total_outcomes']
    == df['num_primary_outcomes'] + df['num_secondary_outcomes'] + df['num_other_outcomes']
).all(), "total_outcomes does not equal the sum of its parts"

print(f"Outcome type distribution (diagnostic):")
for ot in outcome_types_list:
    count = df[f'has_{ot.lower()}_outcome'].sum()
    if count > 0:
        print(f"  {ot}: {count} trials ({count / len(df) * 100:.1f}%)")

print("(End of diagnostic block — see header comment above for scope.)")


RAW LOCATIONS COLUMN DIAGNOSTIC
  Total rows:              541,644
  NaN:                     52,444  (  9.7%)
  Empty list '[]':         52,444  (  9.7%)
  Null-like strings:       0  (  0.0%)

  Parse-status distribution:
    already_list     489,200  ( 90.3%)
    nan               52,444  (  9.7%)

  ✅ No unparseable rows — zero-counts are genuine empties.

  Parse-status % by study type:
protocolSection.contactsLocationsModule.locations  already_list   nan
protocolSection.designModule.studyType                               
EXPANDED_ACCESS                                            57.6  42.4
INTERVENTIONAL                                             91.2   8.8
OBSERVATIONAL                                              87.8  12.2

Parsing outcome data...

=== VERIFICATION (diagnostic; not contractual) ===
Primary outcomes - min: 0, max: 569, mean: 1.96
Total outcomes - min: 0, max: 2334, mean: 6.35
Trials with outcomes: 525881 (97.1%)
Outcome type distribution (diagnostic):
  EFF

In [8]:
# ============================================================================
# CELL 3.5: Therapeutic-Area Binary Flags + Sponsor Regime
# ============================================================================

def _conditions_to_text(v):
    # --- List-likes FIRST: never call pd.isna on these ---
    if isinstance(v, list):
        return ' '.join(str(x).lower() for x in v)
    if isinstance(v, tuple):
        return ' '.join(str(x).lower() for x in v)
    if isinstance(v, np.ndarray):
        return ' '.join(str(x).lower() for x in v.tolist())
    if hasattr(v, 'to_pylist'):
        try:
            return ' '.join(str(x).lower() for x in v.to_pylist())
        except Exception:
            return ''

    # --- Scalars past this point ---
    if v is None:
        return ''
    try:
        if pd.isna(v):
            return ''
    except (ValueError, TypeError):
        # Defensive: unreachable for the shapes Cell 2.1 produces, but
        # guards against exotic array-likes that pd.isna treats array-wise.
        return ' '.join(str(x).lower() for x in v)

    s = str(v).strip()
    if s in ('', '[]', 'nan', 'None', 'null', 'NULL'):
        return ''
    if s.startswith('['):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, list):
                return ' '.join(str(x).lower() for x in parsed)
        except Exception:
            pass
    return s.lower()


_cond_text = df['protocolSection.conditionsModule.conditions'].apply(
    _conditions_to_text
)

for _flag, _keys in _TA_KEYWORDS.items():
    df[_flag] = _cond_text.apply(lambda t: int(any(k in t for k in _keys)))


# ---------------------------------------------------------------------------
# Sponsor-type regime (5 buckets)
#
# CT.gov's protocolSection.identificationModule.organization.class is a
# CLOSED ENUM. We map it explicitly rather than substring-matching, because
# substring matching on a closed vocabulary can silently misclassify (e.g.
# matching 'INDUSTRY' inside an unrelated name) or silently never fire.
#
# The five regimes are:
#
#   industry   — commercial sponsor
#                (enum: INDUSTRY, INDIV)
#   academic   — university, medical school, or hospital sponsor
#                (enum: ACADEMIC, UNKNOWN_ACADEMIC if present)
#   government — federal or non-US-government agency
#                (enum: NIH, FED, OTHER_GOV)
#   network    — cooperative group or trial network
#                (enum: NETWORK)
#   other      — real sponsor CT.gov filed under OTHER without further
#                classification. This is genuinely a catch-all bucket.
#   unknown    — null, empty, AMBIG, UNKNOWN
#
# FIX (Priority #1): 'ACADEMIC' was previously missing from the map and
# silently fell through to 'unknown'. CT.gov uses a distinct ACADEMIC
# enum for university / hospital sponsors, so they were being mislabeled.
# This inflated the 'unknown' bucket and removed a real sponsor category
# from the attribution model's feature matrix.
#
# If CT.gov adds a new enum value that is not in this map, the runtime
# check below will fail loudly rather than silently bucket it as 'unknown'.
# ---------------------------------------------------------------------------
_ENUM_TO_REGIME = {
    # Commercial
    'INDUSTRY':  'industry',
    'INDIV':     'industry',

    # Academic / hospital
    'ACADEMIC':  'academic',

    # Government
    'NIH':       'government',
    'FED':       'government',
    'OTHER_GOV': 'government',

    # Network / cooperative group
    'NETWORK':   'network',

    # Catch-all — CT.gov filed a real sponsor without categorization
    'OTHER':     'other',

    # Explicitly unknown
    'AMBIG':     'unknown',
    'UNKNOWN':   'unknown',
}

# The set of regime strings the map is allowed to produce. If the map is
# extended with a new regime string, this set must be extended in lockstep,
# otherwise the runtime check below will flag every row that resolves to it.
_EXPECTED_REGIME_VALUES = {
    'industry', 'academic', 'government', 'network', 'other', 'unknown',
}


def _derive_sponsor_type(raw):
    """Map CT.gov's organization.class enum to a coarse sponsor regime."""
    if raw is None:
        return 'unknown'
    try:
        if pd.isna(raw):
            return 'unknown'
    except (ValueError, TypeError):
        # Unreachable for scalar input; a list-like slipped past the source
        # projection. Fail loudly rather than silently returning 'unknown'.
        raise ValueError(f"Unexpected array-like sponsor value: {raw!r}")

    s = str(raw).upper().strip().replace('-', '_').replace(' ', '_')
    return _ENUM_TO_REGIME.get(s, 'unknown')


_sponsor_col_regime = 'protocolSection.identificationModule.organization.class'
if _sponsor_col_regime not in df.columns:
    # sponsor_type_regime is the sponsor half of Cell 9.1's regime_bucket,
    # which is strategy_A_label, which is Strategy A's partition key. A
    # silent 'unknown' fallback would collapse Strategy A to a TA-only
    # partition without failing loudly.
    raise RuntimeError(
        f"❌ Required sponsor column {_sponsor_col_regime!r} not found. "
        f"Cell 9.1's regime partition depends on it."
    )

df['sponsor_type_regime'] = df[_sponsor_col_regime].apply(_derive_sponsor_type)


# ---------------------------------------------------------------------------
# Coverage audit — surface any enum value that is NOT in the map.
#
# A value observed in the raw column but missing from _ENUM_TO_REGIME will
# silently become 'unknown' at the row level. This block makes that visible.
# It does NOT raise on missing values (CT.gov may legitimately add new
# enums), but it prints the raw enum and its row count so the operator can
# decide whether to extend the map.
# ---------------------------------------------------------------------------
_raw_enums = (
    df[_sponsor_col_regime]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace('-', '_')
    .str.replace(' ', '_')
)
_observed_enums = set(_raw_enums.unique())
_unmapped_enums = _observed_enums - set(_ENUM_TO_REGIME.keys())

if _unmapped_enums:
    print(f"\n⚠️  {len(_unmapped_enums)} raw enum value(s) are NOT in "
          f"_ENUM_TO_REGIME and will be bucketed as 'unknown':")
    for _e in sorted(_unmapped_enums):
        _n = int((_raw_enums == _e).sum())
        print(f"     {_e:<25} {_n:>9,} rows")
    print("  If any of these are real sponsor types, add them to "
          "_ENUM_TO_REGIME (and _EXPECTED_REGIME_VALUES if a new regime "
          "string is introduced).")

# Sanity: the map must produce only the expected regimes.
_unexpected = set(df['sponsor_type_regime'].unique()) - _EXPECTED_REGIME_VALUES
if _unexpected:
    raise RuntimeError(
        f"❌ sponsor_type_regime produced unexpected values: "
        f"{sorted(_unexpected)}. Update _EXPECTED_REGIME_VALUES to match."
    )


# ---------------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------------
print("\nSponsor-type regime distribution:")
print(df['sponsor_type_regime'].value_counts().to_string())

print("\nTherapeutic-area flag distribution:")
for _flag in _TA_KEYWORDS:
    n = int(df[_flag].sum())
    print(f"  {_flag:<28} {n:>7,}  ({n/len(df)*100:5.1f}%)")

# Overlap diagnostics. Rows with >1 TA flag are silently reduced to a single
# TA by Cell 9.1's np.select (first-match wins). This print is the only place
# that count is visible.
_multi = (df[list(_TA_KEYWORDS)].sum(axis=1) > 1).sum()
_zero = (df[list(_TA_KEYWORDS)].sum(axis=1) == 0).sum()
print(f"\n  Rows with >1 TA flag: {_multi:,}  ({_multi/len(df)*100:.1f}%)")
print(f"  Rows with 0  TA flag: {_zero:,}  ({_zero/len(df)*100:.1f}%)")


Sponsor-type regime distribution:
sponsor_type_regime
other         387925
industry      116799
government     31790
network         4698
unknown          432

Therapeutic-area flag distribution:
  is_oncology                  110,406  ( 20.4%)
  is_cardiovascular             53,404  (  9.9%)
  is_neurology                  43,233  (  8.0%)
  is_infectious_disease         41,441  (  7.7%)
  is_endocrine                  41,644  (  7.7%)
  is_autoimmune                 21,707  (  4.0%)
  is_respiratory                41,159  (  7.6%)
  is_gastrointestinal           25,726  (  4.7%)
  is_psychiatry                 23,517  (  4.3%)
  is_musculoskeletal            29,106  (  5.4%)
  is_dermatology                18,470  (  3.4%)
  is_ophthalmology              10,284  (  1.9%)
  is_renal                      20,628  (  3.8%)
  is_hematology                 22,988  (  4.2%)
  is_pediatrics                 14,760  (  2.7%)
  is_geriatrics                  5,152  (  1.0%)
  is_women_health  

In [9]:
# ============================================================================
# CELL 4: Target variable
# ============================================================================
df['start_date'] = pd.to_datetime(
    df['protocolSection.statusModule.startDateStruct.date'], errors='coerce')
df['completion_date'] = pd.to_datetime(
    df['protocolSection.statusModule.completionDateStruct.date'], errors='coerce')
df['duration_days'] = (df['completion_date'] - df['start_date']).dt.days
df = df[(df['duration_days'] > 0) & (df['duration_days'] < 10950)].copy()
df['duration_days_log'] = np.log1p(df['duration_days'])

In [10]:
# ============================================================================
# CELL 4.1: Business category
# ============================================================================
df['start_year'] = df['start_date'].dt.year.fillna(0).astype(int)
df['duration_category_business'] = pd.cut(
    df['duration_days'],
    bins=[-np.inf, 365, 1095, np.inf],
    labels=['Small (<1yr)', 'Medium (1-3yr)', 'Large (>3yr)'],
)
df['duration_category_business_num'] = (
    df['duration_category_business']
    .map({'Small (<1yr)': 0, 'Medium (1-3yr)': 1, 'Large (>3yr)': 2})
)

In [11]:
# ============================================================================
# CELL 5: Feature Engineering
# ============================================================================

# ============ PART 1: PHASE COMBINATION FEATURES ============

def get_phase_set(phase_data):
    """Extract clean phase numbers from phase data"""
    # Array-likes first — never call pd.isna on these
    if isinstance(phase_data, np.ndarray):
        phase_data = phase_data.tolist()
    if hasattr(phase_data, 'to_pylist'):
        try:
            phase_data = phase_data.to_pylist()
        except Exception:
            return set()

    # Scalar NaN check — guarded against array return
    if phase_data is None:
        return set()
    try:
        if pd.isna(phase_data):
            return set()
    except (ValueError, TypeError):
        return set()

    if isinstance(phase_data, list):
        clean_phases = []
        for p in phase_data:
            p_str = str(p).upper()
            import re
            match = re.search(r'PHASE(\d+)', p_str)
            if match:
                clean_phases.append(int(match.group(1)))
            elif 'EARLY_PHASE1' in p_str:
                clean_phases.append(0)
        return set(clean_phases)
    elif isinstance(phase_data, str):
        p_str = phase_data.upper()
        import re
        matches = re.findall(r'PHASE(\d+)', p_str)
        return set([int(m) for m in matches])
    return set()

df['phase_set'] = df['protocolSection.designModule.phases'].apply(get_phase_set)
# ---  derive phase_score from phase_set --------------------------------
# Convention:
#   0 -> no phase           (phase_set is empty)
#   1 -> Phase 1 / Early Phase 1
#   2 -> Phase 2
#   3 -> Phase 3
#   4 -> Phase 4
# For mixed-phase trials, the score is the MAX phase present.
def _phase_set_to_score(ps):
    if not ps:
        return 0
    if 4 in ps: return 4
    if 3 in ps: return 3
    if 2 in ps: return 2
    if 1 in ps: return 1
    if 0 in ps: return 1   # Early Phase 1 maps to score 1 (same as Phase 1)
    return 0

df['phase_score'] = df['phase_set'].apply(_phase_set_to_score)

df['phase_count'] = df['phase_set'].apply(len)

for phase in [0, 1, 2, 3, 4]:
    df[f'has_phase{phase}'] = df['phase_set'].apply(lambda x: phase in x)

phase_combinations = [
    (1, 2), (1, 2, 3), (1, 2, 3, 4),
    (2, 3), (2, 3, 4),
    (3, 4),
    (1,), (2,), (3,), (4,),
]
for combo in phase_combinations:
    combo_str = '-'.join([str(p) for p in combo])
    df[f'phase_combo_{combo_str}'] = df['phase_set'].apply(
        lambda x: all(p in x for p in combo) and len(x) == len(combo)
    )

def is_consecutive(phase_set):
    if len(phase_set) <= 1:
        return False
    sorted_phases = sorted(phase_set)
    return all(sorted_phases[i+1] - sorted_phases[i] == 1 for i in range(len(sorted_phases)-1))

df['phase_consecutive'] = df['phase_set'].apply(is_consecutive)

def get_phase_gaps(phase_set):
    sorted_phases = sorted(phase_set)
    if len(sorted_phases) <= 1:
        return 0
    gaps = [sorted_phases[i+1] - sorted_phases[i] - 1 for i in range(len(sorted_phases)-1)]
    return max(gaps) if gaps else 0

df['phase_gaps'] = df['phase_set'].apply(get_phase_gaps)

def get_phase_coverage(phase_set):
    if not phase_set:
        return 0
    max_phase = max(phase_set)
    expected = set(range(max_phase + 1))
    return len(phase_set & expected) / len(expected) if expected else 0

df['phase_coverage'] = df['phase_set'].apply(get_phase_coverage)

def get_progression_score(phase_set):
    if 4 in phase_set:
        return 5
    elif 3 in phase_set:
        return 4
    elif 2 in phase_set:
        return 3
    elif 1 in phase_set:
        return 2
    elif 0 in phase_set:
        return 1
    return 0

df['phase_progression_score'] = df['phase_set'].apply(get_progression_score)

df['is_early_phase'] = df['phase_set'].apply(
    lambda x: any(p in x for p in [0, 1]) and not any(p in x for p in [3, 4])
)
df['is_late_phase'] = df['phase_set'].apply(
    lambda x: any(p in x for p in [3, 4]) and not any(p in x for p in [0, 1])
)
df['is_mixed_phase'] = df['phase_set'].apply(
    lambda x: any(p in x for p in [0, 1]) and any(p in x for p in [3, 4])
)


# ============ PART 2: FUNDING SOURCE FEATURES ============

def get_funding_sources(sponsor_data):
    if pd.isna(sponsor_data):
        return ['UNKNOWN']
    sponsor_str = str(sponsor_data).upper()
    funders = []
    if any(key in sponsor_str for key in ['INDUSTRY', 'PHARMA', 'PHARMACEUTICAL', 'BIOTECH', 'MEDICAL DEVICE', 'CORPORATE']):
        funders.append('INDUSTRY')
    if any(key in sponsor_str for key in ['NIH', 'NATIONAL INSTITUTES', 'FEDERAL', 'GOVERNMENT', 'DEPARTMENT OF', 'CDC', 'FDA', 'NSF', 'DOD', 'VA']):
        funders.append('GOVERNMENT')
    if any(key in sponsor_str for key in ['UNIVERSITY', 'COLLEGE', 'ACADEMIC', 'MEDICAL SCHOOL', 'HOSPITAL', 'INSTITUTE']):
        funders.append('ACADEMIC')
    if any(key in sponsor_str for key in ['FOUNDATION', 'NON-PROFIT', 'NONPROFIT', 'ASSOCIATION', 'SOCIETY', 'CHARITY']):
        funders.append('NONPROFIT')
    if any(key in sponsor_str for key in ['WHO', 'WORLD HEALTH', 'UN', 'UNITED NATIONS', 'EUROPEAN', 'GLOBAL']):
        funders.append('INTERNATIONAL')
    return funders if funders else ['OTHER']

sponsor_col = 'protocolSection.identificationModule.organization.class'
df['funding_sources'] = df[sponsor_col].apply(get_funding_sources)

funding_types = ['INDUSTRY', 'GOVERNMENT', 'ACADEMIC', 'NONPROFIT', 'INTERNATIONAL', 'OTHER', 'UNKNOWN']
for ft in funding_types:
    df[f'funding_{ft.lower()}'] = df['funding_sources'].apply(lambda x: ft in x)

df['funding_source_count'] = df['funding_sources'].apply(len)

def get_funding_diversity(sources):
    if not sources:
        return 0
    counts = {}
    for s in sources:
        counts[s] = counts.get(s, 0) + 1
    total = sum(counts.values())
    return 1 - sum((count/total)**2 for count in counts.values())

df['funding_diversity'] = df['funding_sources'].apply(get_funding_diversity)

df['is_industry_funded'] = df['funding_industry']
df['is_public_funded'] = df['funding_government'] | df['funding_academic']
df['is_nonprofit_funded'] = df['funding_nonprofit']
df['funding_industry_academic'] = df['funding_industry'] & df['funding_academic']
df['funding_gov_academic'] = df['funding_government'] & df['funding_academic']
df['funding_industry_gov'] = df['funding_industry'] & df['funding_government']

def get_primary_funding(sources):
    if not sources:
        return 'UNKNOWN'
    priority = ['INDUSTRY', 'GOVERNMENT', 'NONPROFIT', 'ACADEMIC', 'INTERNATIONAL', 'OTHER']
    for p in priority:
        if p in sources:
            return p
    return 'UNKNOWN'

df['primary_funding'] = df['funding_sources'].apply(get_primary_funding)


# ============ PART 3: OUTCOME COMPLEXITY FEATURES ============

df['outcome_complexity_primary'] = df['protocolSection.outcomesModule.primaryOutcomes'].apply(
    lambda x: get_outcome_complexity(x)
)
df['outcome_complexity_secondary'] = df['protocolSection.outcomesModule.secondaryOutcomes'].apply(
    lambda x: get_outcome_complexity(x)
)

def get_outcome_balance(primary, secondary):
    if primary == 0 and secondary == 0:
        return 0
    if primary == 0:
        return 1
    return min(secondary / primary, 3)

df['outcome_balance'] = df.apply(
    lambda x: get_outcome_balance(x['num_primary_outcomes'], x['num_secondary_outcomes']),
    axis=1
)

# ============ PART 4: ORIGINAL FEATURES ============

# Operational features
df['num_sites'] = df['protocolSection.contactsLocationsModule.locations'].apply(count_sites)
df['num_countries'] = df['protocolSection.contactsLocationsModule.locations'].apply(count_countries)
df['us_sites'] = df['protocolSection.contactsLocationsModule.locations'].apply(count_us_sites)
df['us_percentage'] = df['us_sites'] / df['num_sites']
df['us_percentage'] = df['us_percentage'].replace([np.inf, -np.inf], 0).fillna(0)
df['has_results'] = df['hasResults'].apply(lambda x: 1 if str(x).lower() in ['yes', 'true', '1'] else 0)
df['enrollment_count'] = df['protocolSection.designModule.enrollmentInfo.count']
df['has_locations'] = (
    df['protocolSection.contactsLocationsModule.locations']
    .notna()
    .astype(int)
)

# ---------------------------------------------------------------------------
# 5a.  PER-BRANCH num_sites DIAGNOSTIC
# ---------------------------------------------------------------------------
# `num_sites` is computed globally. But downstream, we split into
# interventional vs observational branches. If `num_sites` is constant
# (or near-constant) within one branch, the decision tree will silently
# refuse to split on it — producing clusters with misleading site labels
# (e.g. "Single-Site" on a cluster whose mean is 0.0). This block flags
# that risk *before* the tree sees the feature.

print("\n" + "=" * 70)
print("NUM_SITES DIAGNOSTIC (per branch, before tree fitting)")
print("=" * 70)

# Derive study_type early — it is needed for the diagnostic and is
# identical to the version computed later in this cell.
_study_type_diag = df['protocolSection.designModule.studyType'].apply(get_study_type)
df['study_type'] = _study_type_diag

if 'study_type' in df.columns:
    _ns_stats = df.groupby('study_type')['num_sites'].describe()
    print("\n  num_sites summary by study_type:")
    print(_ns_stats.round(2).to_string())

    _ns_var = df.groupby('study_type')['num_sites'].var()
    print("\n  Variance of num_sites by study_type:")
    for st, v in _ns_var.items():
        flag = "⚠️  DEGENERATE" if (pd.isna(v) or v < 1e-6) else "✅"
        print(f"    {st:<20}  var = {v:>10.4f}   {flag}")

    _ns_zero_rate = df.groupby('study_type')['num_sites'].apply(
        lambda s: (s == 0).mean() * 100
    )
    print("\n  Fraction of rows with num_sites == 0, by study_type:")
    for st, r in _ns_zero_rate.items():
        flag = "⚠️  MOSTLY-ZERO" if r > 50 else "✅"
        print(f"    {st:<20}  {r:>5.1f}%   {flag}")

    _degenerate_branches = [
        st for st, v in _ns_var.items()
        if pd.isna(v) or v < 1e-6
    ]
    if _degenerate_branches:
        print(f"\n  🚨 WARNING: num_sites is CONSTANT in branch(es): "
              f"{_degenerate_branches}")
        print("     The decision tree will NOT split on num_sites for these "
              "branches.")
        print("     Cluster descriptors like 'Single-Site' / 'Multi-Site' "
              "will be meaningless there.")
    else:
        print("\n  ✅ num_sites has non-zero variance in every branch.")
else:
    print("  (study_type not yet derived — skipping per-branch check.)")

print("=" * 70)

# Design features
df['primary_purpose'] = df['protocolSection.designModule.designInfo.primaryPurpose'].fillna('UNKNOWN')
df['masking'] = df['protocolSection.designModule.designInfo.maskingInfo.masking'].fillna('UNKNOWN')
df['time_perspective'] = df['protocolSection.designModule.designInfo.timePerspective'].fillna('UNKNOWN')
df['num_arm_groups'] = df['protocolSection.armsInterventionsModule.armGroups'].apply(count_items)

# Scope features
df['num_conditions'] = df['protocolSection.conditionsModule.conditions'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Eligibility features
df['min_age'] = df['protocolSection.eligibilityModule.minimumAge'].apply(extract_age_simple)
df['max_age'] = df['protocolSection.eligibilityModule.maximumAge'].apply(extract_age_simple)
df['age_range'] = (df['max_age'] - df['min_age']).clip(lower=0)
df['eligibility_word_count'] = df['protocolSection.eligibilityModule.eligibilityCriteria'].apply(count_criteria_words)

# Enhanced complexity score
df['complexity_score'] = (
    df['num_sites'] / 10 +
    df['num_conditions'] * 2 +
    df['phase_count'] * 3 +
    df['num_arm_groups'] * 2 +
    df['num_primary_outcomes'] * 1.5 +
    df['num_secondary_outcomes'] * 0.5 +
    df['funding_source_count'] * 1
)

# Planned duration
df['planned_duration_years'] = df['protocolSection.designModule.targetDuration'].apply(extract_years)
df['planned_duration_days'] = df['planned_duration_years'] * 365.25

# Log enrollment
df['log_enrollment'] = np.log1p(df['enrollment_count'])

# ============ PART 5: PRINT FEATURE SUMMARY ============

print("\n" + "="*60)
print("ENHANCED FEATURE ENGINEERING COMPLETE")
print("="*60)

phase_features = [col for col in df.columns if col.startswith(('phase_', 'has_phase'))]
funding_features = [col for col in df.columns if col.startswith(('funding_', 'primary_funding'))]
outcome_features = [col for col in df.columns if col.startswith(('num_', 'outcome_', 'has_')) and col.endswith('_outcome')]

print(f"\n✅ Phase Combination Features: {len(phase_features)} new features")
print(f"   Examples: phase_count, phase_progression_score, is_early_phase, phase_combo_1-2-3")

print(f"\n✅ Funding Source Features: {len(funding_features)} new features")
print(f"   Examples: funding_source_count, funding_diversity, is_industry_funded, primary_funding")

print(f"\n✅ Outcome Complexity Features: {len(outcome_features)} new features")
print(f"   Examples: total_outcomes, outcome_complexity_total, outcome_type_count, has_efficacy_outcome")

print(f"\n✅ Total enhanced features: {len(phase_features) + len(funding_features) + len(outcome_features)}")


NUM_SITES DIAGNOSTIC (per branch, before tree fitting)

  num_sites summary by study_type:
                    count   mean    std  min  25%  50%    75%     max
study_type                                                           
EXPANDED_ACCESS      40.0  37.08  87.79  0.0  1.0  1.5  15.25   453.0
INTERVENTIONAL   136579.0   6.22  25.45  0.0  1.0  1.0   2.00  1191.0
OBSERVATIONAL     35302.0   4.35  39.33  0.0  1.0  1.0   1.00  3510.0

  Variance of num_sites by study_type:
    EXPANDED_ACCESS       var =  7707.7122   ✅
    INTERVENTIONAL        var =   647.8829   ✅
    OBSERVATIONAL         var =  1546.5275   ✅

  Fraction of rows with num_sites == 0, by study_type:
    EXPANDED_ACCESS        12.5%   ✅
    INTERVENTIONAL         12.3%   ✅
    OBSERVATIONAL          18.6%   ✅

  ✅ num_sites has non-zero variance in every branch.

ENHANCED FEATURE ENGINEERING COMPLETE

✅ Phase Combination Features: 22 new features
   Examples: phase_count, phase_progression_score, is_early_phase, pha

In [12]:
# ============================================================================
# CELL 5.1: Feature Transformation Pipeline
# ============================================================================

print("\n" + "=" * 70)
print("FEATURE TRANSFORMATION PIPELINE")
print("=" * 70)


def _safe_parse_list_conditions(val):
    # --- List-likes FIRST: never call pd.isna on these ---
    if isinstance(val, list):
        return val
    if isinstance(val, tuple):
        return list(val)
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, 'to_pylist'):
        try:
            return val.to_pylist()
        except Exception:
            return []

    # --- Scalars past this point ---
    if val is None:
        return []
    try:
        if pd.isna(val):
            return []
    except (ValueError, TypeError):
        return []

    s = str(val).strip()
    if s in ('', '[]', 'nan', 'None'):
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    try:
        parsed = json.loads(s.replace("'", '"'))
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    return []


if 'protocolSection.conditionsModule.conditions' in df.columns:
    df['num_conditions'] = df['protocolSection.conditionsModule.conditions'].apply(
        lambda v: len(_safe_parse_list_conditions(v))
    )
    print(f"  ✅ Re-parsed num_conditions "
          f"(min={df['num_conditions'].min()}, "
          f"max={df['num_conditions'].max()}, "
          f"mean={df['num_conditions'].mean():.2f})")
else:
    print("  ⚠️ conditions column not found — num_conditions left unchanged")


HEAVY_TAILED_NUMERICS = ['enrollment_count', 'eligibility_word_count']
for col in HEAVY_TAILED_NUMERICS:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0).fillna(0))
        print(f"  ✅ Created {col}_log")

if 'complexity_score' in df.columns:
    df['complexity_score_log'] = np.log1p(
        df['complexity_score'].clip(lower=0).fillna(0)
    )
    print(f"  ✅ Created complexity_score_log")


CATEGORICALS_TO_FREQ_ENCODE = [
    'masking',
    'protocolSection.sponsorCollaboratorsModule.leadSponsor.class',
    'protocolSection.designModule.designInfo.primaryPurpose',
]
for col in CATEGORICALS_TO_FREQ_ENCODE:
    if col in df.columns:
        freq_map = df[col].fillna('UNKNOWN').value_counts(normalize=True)
        df[f'{col}_freq'] = df[col].fillna('UNKNOWN').map(freq_map)
        print(f"  ✅ Created {col}_freq")


DEDUP_PAIRS = [
    ('masking', 'masking_freq'),
    ('protocolSection.sponsorCollaboratorsModule.leadSponsor.class',
     'protocolSection.sponsorCollaboratorsModule.leadSponsor.class_freq'),
    ('protocolSection.designModule.designInfo.primaryPurpose',
     'protocolSection.designModule.designInfo.primaryPurpose_freq'),
]
for raw_col, freq_col in DEDUP_PAIRS:
    if raw_col in df.columns and freq_col in df.columns:
        df = df.drop(columns=[raw_col])
        print(f"  🗑️ Dropped redundant raw column '{raw_col}'")


# ---- SAFETY: guarantee duration column is NOT in numeric pool ---- #
# We keep it in the df (needed for validation) but downstream selectors
# explicitly exclude it via `exclude_patterns`.
assert 'duration_days_log' in df.columns, "duration target missing"

print("\n✅ Feature transformation pipeline complete.")
print("=" * 70)


FEATURE TRANSFORMATION PIPELINE
  ✅ Re-parsed num_conditions (min=1, max=174, mean=1.61)
  ✅ Created enrollment_count_log
  ✅ Created eligibility_word_count_log
  ✅ Created complexity_score_log
  ✅ Created masking_freq
  ✅ Created protocolSection.sponsorCollaboratorsModule.leadSponsor.class_freq
  ✅ Created protocolSection.designModule.designInfo.primaryPurpose_freq
  🗑️ Dropped redundant raw column 'masking'
  🗑️ Dropped redundant raw column 'protocolSection.sponsorCollaboratorsModule.leadSponsor.class'
  🗑️ Dropped redundant raw column 'protocolSection.designModule.designInfo.primaryPurpose'

✅ Feature transformation pipeline complete.


In [13]:
# ============================================================================
# CELL 5.2: ELIGIBILITY TEXT — ENGINEERED FEATURES
# ============================================================================
# Derives eight interpretable scalar features from the raw eligibility
# criteria text via regex and counting. No fitting, no vectorization, no
# model.
#
# Output columns:
#   elig_text_len, elig_text_word_count, elig_text_sentence_count,
#   elig_text_negation_count, elig_text_numeric_token_count,
#   elig_text_prior_therapy_mention, elig_text_placebo_mention,
#   elig_text_age_bound_present
#
# These columns exist for interpretability (SHAP attributions like
# "elig_text_len" are readable; "elig_text_svd_047" is not) and because
# they are free — no fit, no persistence, no inference-time contract.
#
# FITTED TEXT REPRESENTATIONS ARE NOT PRODUCED HERE.
# An earlier pipeline variant fit TF-IDF + TruncatedSVD (and optionally
# sentence-transformer embeddings) on the train fold. An internal ablation
# found the SVD dims improved the global model only marginally, and did
# not improve the deployed routed system at all — the cluster specialists
# are fit on the structural pool by construction, so no fitted text
# representation reaches the routed track. The cost (persistence, an
# inference-time normalization contract, opaque SHAP attributions, a
# wider handoff pool) outweighed the benefit, so the fit was removed.
#
# The schema for the fitted families is retained — see Cell 8.1's
# text_representation branches and the four-family inventory — so a future
# variant can re-add the fit without changing downstream consumers. Until
# then, the svd_text and embedding_text families are always empty and
# text_representation resolves to 'engineered_only'.
# ============================================================================

import re

import numpy as np
import pandas as pd

print("\n" + "=" * 70)
print("ELIGIBILITY TEXT — ENGINEERED FEATURES")
print("=" * 70)

_ELIG_COL = 'protocolSection.eligibilityModule.eligibilityCriteria'

if _ELIG_COL not in df.columns:
    raise RuntimeError(
        f"❌ Eligibility column {_ELIG_COL!r} not found in df. "
        f"Provide the correct parquet."
    )

# ---------------------------------------------------------------------------
# 1. Normalize the raw text (defensive: list/dict/NaN all collapse to str)
# ---------------------------------------------------------------------------
def _to_clean_text(v):
    if isinstance(v, list):
        return ' '.join(str(x) for x in v)
    if isinstance(v, tuple):
        return ' '.join(str(x) for x in v)
    if isinstance(v, np.ndarray):
        return ' '.join(str(x) for x in v.tolist())
    if hasattr(v, 'to_pylist'):
        try:
            return ' '.join(str(x) for x in v.to_pylist())
        except Exception:
            return ''
    if v is None:
        return ''
    try:
        if pd.isna(v):
            return ''
    except (ValueError, TypeError):
        return ''
    return str(v)

_elig_raw = df[_ELIG_COL].apply(_to_clean_text)

# Lowercase + strip HTML/whitespace for regex matching.
def _normalize_text(t):
    t = t.lower()
    t = re.sub(r'<[^>]+>', ' ', t)          # strip HTML tags
    t = re.sub(r'[^a-z0-9\.\,\-\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

_elig_clean = _elig_raw.apply(_normalize_text)

_n_empty = int((_elig_clean.str.len() == 0).sum())
print(f"  Rows:            {len(df):,}")
print(f"  Empty criteria:  {_n_empty:,}  ({_n_empty/len(df)*100:5.1f}%)")

# ---------------------------------------------------------------------------
# 2. Engineered features
# ---------------------------------------------------------------------------
print("\n  Deriving interpretable text features ...")

_NEGATION_TOKENS = (
    r'\bno\b', r'\bnot\b', r'\bwithout\b', r'\bexclude\b',
    r'\bexcluded\b', r'\bnegative\b', r'\babsence\b',
)
_NEGATION_RE = re.compile('|'.join(_NEGATION_TOKENS))
_NUMERIC_RE  = re.compile(r'\b\d+(\.\d+)?\b')
_SENT_RE     = re.compile(r'[\.\;\n]+')

df['elig_text_len']              = _elig_raw.str.len().astype(int)
df['elig_text_word_count']       = _elig_clean.str.split().apply(len).astype(int)
df['elig_text_sentence_count']   = _elig_raw.apply(
    lambda t: max(1, len(_SENT_RE.split(t))) if t else 0
).astype(int)
df['elig_text_negation_count']   = _elig_clean.apply(
    lambda t: len(_NEGATION_RE.findall(t))
).astype(int)
df['elig_text_numeric_token_count'] = _elig_clean.apply(
    lambda t: len(_NUMERIC_RE.findall(t))
).astype(int)

df['elig_text_prior_therapy_mention'] = _elig_clean.apply(
    lambda t: int(any(k in t for k in (
        'prior therapy', 'previous treatment', 'prior treatment',
        'prior systemic', 'failed therapy', 'refractory',
    )))
).astype(int)

df['elig_text_placebo_mention'] = _elig_clean.apply(
    lambda t: int('placebo' in t)
).astype(int)

df['elig_text_age_bound_present'] = _elig_clean.apply(
    lambda t: int(('years' in t or 'year' in t or 'age' in t))
).astype(int)

_ENGINEERED_TEXT_COLS = [
    'elig_text_len',
    'elig_text_word_count',
    'elig_text_sentence_count',
    'elig_text_negation_count',
    'elig_text_numeric_token_count',
    'elig_text_prior_therapy_mention',
    'elig_text_placebo_mention',
    'elig_text_age_bound_present',
]

# ---------------------------------------------------------------------------
# 3. Sanity — all numeric, no NaN
# ---------------------------------------------------------------------------
for _c in _ENGINEERED_TEXT_COLS:
    assert pd.api.types.is_numeric_dtype(df[_c]), (
        f"❌ Engineered text feature {_c!r} is not numeric "
        f"(dtype={df[_c].dtype})"
    )
    if df[_c].isna().any():
        raise RuntimeError(
            f"❌ Engineered text feature {_c!r} contains NaN "
            f"({int(df[_c].isna().sum()):,} rows)."
        )

print(f"  ✅ {len(_ENGINEERED_TEXT_COLS)} engineered text features created:")
for _c in _ENGINEERED_TEXT_COLS:
    _mn = df[_c].min()
    _mx = df[_c].max()
    _med = df[_c].median()
    print(f"     {_c:<40} min={_mn:>7.0f}  median={_med:>8.1f}  max={_mx:>8.0f}")

# ---------------------------------------------------------------------------
# 4. Release the transient frames
# ---------------------------------------------------------------------------
del _elig_raw, _elig_clean
import gc
gc.collect()

print("=" * 70)


ELIGIBILITY TEXT — ENGINEERED FEATURES
  Rows:            171,921
  Empty criteria:  32  (  0.0%)

  Deriving interpretable text features ...
  ✅ 8 engineered text features created:
     elig_text_len                            min=      0  median=   762.0  max=   19987
     elig_text_word_count                     min=      0  median=   104.0  max=    2226
     elig_text_sentence_count                 min=      0  median=    14.0  max=     322
     elig_text_negation_count                 min=      0  median=     1.0  max=      50
     elig_text_numeric_token_count            min=      0  median=     5.0  max=     187
     elig_text_prior_therapy_mention          min=      0  median=     0.0  max=       1
     elig_text_placebo_mention                min=      0  median=     0.0  max=       1
     elig_text_age_bound_present              min=      0  median=     1.0  max=       1


In [14]:
# ============================================================================
# CELL 5.3: TEMPORAL PROXY — years_since_registration
# ============================================================================
# Replaces raw `start_year` with a drift-invariant proxy. The reference date
# is FROZEN the first time this cell runs, stored in the shared bundle, and
# read back on every subsequent run so that retraining a year later does NOT
# shift the scalar. Notebooks 03 and 04 apply the identical transformation.
#
# Reference-date handling:
#   1. If cluster_data/shared/temporal_reference.json exists, READ the frozen
#      reference_date from it. Do NOT recompute. This preserves comparability
#      across reruns and across retrained models.
#   2. If it does not exist, FREEZE it now:
#        - guard against implausible future start_dates using TODAY as the
#          plausibility ceiling (a 2099 row must not become the reference)
#        - take the max of the surviving train-fold start_dates
#        - persist it to temporal_reference.json
#
# Adds columns:
#   years_since_registration       (float, >= 0)
#   years_since_registration_log   (log1p, for heavy-tailed stability)
# ============================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("\n" + "=" * 70)
print("TEMPORAL PROXY — years_since_registration")
print("=" * 70)

if '_split' not in df.columns:
    raise RuntimeError("❌ df['_split'] not found. Cell 2.2 must run first.")

if 'start_date' not in df.columns:
    raise RuntimeError(
        "❌ df['start_date'] not found. Cell 4 must run before Cell 5.1c."
    )

_shared_root = Path('cluster_data/shared')
_shared_root.mkdir(parents=True, exist_ok=True)
_temporal_ref_path = _shared_root / 'temporal_reference.json'


# ---------------------------------------------------------------------------
# 1. Decide the reference date:
#       - READ it from disk if a prior run froze one
#       - otherwise FREEZE it now, with a plausibility guard
# ---------------------------------------------------------------------------
_train_dates_all = pd.to_datetime(
    df.loc[df['_split'] == 'train', 'start_date'], errors='coerce'
).dropna()

if len(_train_dates_all) == 0:
    raise RuntimeError(
        "❌ No valid start_date values in the train fold. "
        "Cannot derive years_since_registration."
    )

if _temporal_ref_path.exists():
    # -------------------------------------------------------------------
    # Path A: a frozen reference already exists — reuse it verbatim.
    # -------------------------------------------------------------------
    with open(_temporal_ref_path) as f:
        _temporal_meta_in = json.load(f)

    if 'reference_date' not in _temporal_meta_in:
        raise RuntimeError(
            f"❌ {_temporal_ref_path} exists but has no 'reference_date'. "
            f"Delete it and rerun this cell to freeze a fresh one."
        )

    _temporal_reference_date = pd.Timestamp(_temporal_meta_in['reference_date'])
    _reference_source = 'frozen_on_disk'

    print(f"  Reference date (read from {_temporal_ref_path.name}): "
          f"{_temporal_reference_date.date()}")
    print(f"  ℹ️  Frozen reference reused — NOT recomputed from this data. "
          f"This keeps scalars comparable across reruns.")

else:
    # -------------------------------------------------------------------
    # Path B: no frozen reference — freeze one now, with a plausibility
    # guard. Today's date is the ceiling, not a hardcoded constant, so
    # the guard never needs manual maintenance.
    # -------------------------------------------------------------------
    _PLAUSIBLE_MAX = pd.Timestamp.today().normalize()

    _n_before = len(_train_dates_all)
    _train_dates_ok = _train_dates_all[_train_dates_all <= _PLAUSIBLE_MAX]
    _n_dropped = _n_before - len(_train_dates_ok)

    if len(_train_dates_ok) == 0:
        raise RuntimeError(
            f"❌ All {_n_before:,} train-fold start_date values are after "
            f"today ({_PLAUSIBLE_MAX.date()}). This is a data-quality "
            f"problem — inspect `start_date` before proceeding."
        )

    if _n_dropped > 0:
        _bad_years = sorted(
            _train_dates_all[_train_dates_all > _PLAUSIBLE_MAX]
            .dt.year.unique().tolist()
        )
        print(f"  ⚠️  Dropped {_n_dropped:,} implausible start_date row(s) "
              f"from the reference computation.")
        print(f"      Offending year(s): {_bad_years}")
        print(f"      Ceiling applied:   {_PLAUSIBLE_MAX.date()} (today)")

    _temporal_reference_date = _train_dates_ok.max()
    _reference_source = 'frozen_now'

    print(f"  Reference date (train max start_date, plausibility-guarded): "
          f"{_temporal_reference_date.date()}")
    print(f"  ℹ️  Freezing this date to {_temporal_ref_path.name}. "
          f"Subsequent runs will read it back, not recompute it.")


# ---------------------------------------------------------------------------
# 2. Compute the proxy against the frozen reference date
# ---------------------------------------------------------------------------
_start_dates_all = pd.to_datetime(df['start_date'], errors='coerce')

_years_since = (
    (_temporal_reference_date - _start_dates_all).dt.days / 365.25
)

# Rows with no start_date get the median of the train fold (so they don't
# silently become huge outliers).
_train_median = _years_since[df['_split'] == 'train'].median()
if pd.isna(_train_median):
    _train_median = 0.0

_years_since = _years_since.fillna(_train_median)
_years_since = _years_since.clip(lower=0.0)

df['years_since_registration']     = _years_since.astype(float)
df['years_since_registration_log'] = np.log1p(
    df['years_since_registration']
).astype(float)


# ---------------------------------------------------------------------------
# 3. Sanity
# ---------------------------------------------------------------------------
assert df['years_since_registration'].notna().all()
assert (df['years_since_registration'] >= 0).all()

_q = df['years_since_registration'].quantile([0.01, 0.5, 0.99]).round(2)
print(f"  years_since_registration — "
      f"p01={_q.iloc[0]}  median={_q.iloc[1]}  p99={_q.iloc[2]}")
print(f"  ✅ {len(df):,} rows populated")

# Plausibility sanity on the DERIVED scalar: a median near the reference
# date itself usually means the reference is wrong (a single future-dated
# row took over the max). Guard with a soft warning rather than an assert,
# because some registries legitimately skew old.
if _q.iloc[1] > 60:
    print(f"  ⚠️  median years_since_registration = {_q.iloc[1]:.1f} — "
          f"unusually large. Verify that the frozen reference_date "
          f"({_temporal_reference_date.date()}) is plausible.")


# ---------------------------------------------------------------------------
# 4. Persist the reference date on the first run only
# ---------------------------------------------------------------------------
if _reference_source == 'frozen_now':
    _temporal_meta = {
        'reference_date':         _temporal_reference_date.isoformat(),
        'derived_from':           'start_date',
        'fit_scope':              'train_only',
        'plausibility_ceiling':   _PLAUSIBLE_MAX.isoformat(),
        'n_train_dates_before':   int(_n_before),
        'n_train_dates_dropped':  int(_n_dropped),
        'fill_value_for_missing': float(_train_median),
        'columns': [
            'years_since_registration',
            'years_since_registration_log',
        ],
        'note': (
            'reference_date is FROZEN at the max start_date observed in the '
            'training fold, after excluding start_dates after the run-time '
            'plausibility ceiling (today). At inference time, compute '
            'years_since_registration = (reference_date - start_date) / 365.25. '
            'Do NOT recompute reference_date from new data — read this file.'
        ),
    }
    with open(_temporal_ref_path, 'w') as f:
        json.dump(_temporal_meta, f, indent=2)

    print(f"  ✅ Froze and saved {_temporal_ref_path}")
else:
    print(f"  ✅ Reused frozen reference from {_temporal_ref_path}")

print("=" * 70)


TEMPORAL PROXY — years_since_registration
  Reference date (read from temporal_reference.json): 2026-10-01
  ℹ️  Frozen reference reused — NOT recomputed from this data. This keeps scalars comparable across reruns.
  years_since_registration — p01=0.08  median=15.59  p99=30.33
  ✅ 171,921 rows populated
  ✅ Reused frozen reference from cluster_data\shared\temporal_reference.json


In [15]:
# ============================================================================
# CELL 5.4: df_full alias
# ============================================================================
df_full = df
print(f"✅ df_full alias: {df_full.shape[0]:,} rows × {df_full.shape[1]} columns")

✅ df_full alias: 171,921 rows × 145 columns


In [16]:
# ============================================================================
# CELL 5.5: REFERENCE FEATURE POOL
# ============================================================================
# For peer-group benchmarking, we do NOT prune features. Every interpretable
# feature is a candidate explanation for why a trial took as long as it did.
# The attribution model in Cell 13.2 handles redundancy internally.
#
# Outputs:
#   _AUDIT_FEATURE_POOL — the feature list used for attribution and for
#                         describing peer groups.
#   Persisted to cluster_data/shared/audit_feature_pool.json.

import json
from pathlib import Path

print("\n" + "=" * 70)
print("REFERENCE FEATURE POOL (unpruned)")
print("=" * 70)

_FORBIDDEN_POOL = {
    'duration_days', 'duration_days_log',
    'duration_category', 'duration_category_business',
    'duration_category_business_num',
    'start_date', 'completion_date',
    'start_year',                       # replaced by years_since_registration
    'hasResults', 'has_results',
    'planned_duration_days', 'planned_duration_days_clean',
    'planned_duration_days_original', 'planned_actual_ratio',
    'cluster', 'cluster_local', 'cluster_merged',
    'tier', 'strategy_A_label', 'strategy_B_label',
    'regime_bucket',
    '_split', '_is_train', '_global_resid', '_loc_missing',
    'phase_set',
}

_TEXT_COLS_STRUCT_FLAGS = sorted(
    c for c in df.columns
    if (c.startswith(('has_', 'kw_', 'outcome_has_'))
        or c.endswith('_text_len'))
    and c not in {'has_results', 'has_locations',
                  'has_phase0', 'has_phase1', 'has_phase2',
                  'has_phase3', 'has_phase4'}
)
_TEXT_COLS_ENGINEERED = sorted(
    c for c in df.columns
    if c.startswith('elig_text_')
    and not c.startswith('elig_text_svd_')
    and not c.startswith('elig_text_emb_')
)
_TEXT_COLS_SVD = sorted(c for c in df.columns if c.startswith('elig_text_svd_'))
_TEXT_COLS_EMB = sorted(c for c in df.columns if c.startswith('elig_text_emb_'))

_TEXT_COLS_ALL = (_TEXT_COLS_STRUCT_FLAGS + _TEXT_COLS_ENGINEERED
                  + _TEXT_COLS_SVD + _TEXT_COLS_EMB)

_AUDIT_FEATURE_POOL = []
for _c in df.columns:
    if _c in _FORBIDDEN_POOL:          continue
    if _c.startswith('_'):              continue
    if _c.startswith('duration'):       continue
    if pd.api.types.is_numeric_dtype(df[_c]) or pd.api.types.is_bool_dtype(df[_c]):
        _AUDIT_FEATURE_POOL.append(_c)

_AUDIT_FEATURE_POOL = sorted(set(_AUDIT_FEATURE_POOL))

print(f"  Audit feature pool: {len(_AUDIT_FEATURE_POOL)} features")
print(f"    structural/bool:  {len([f for f in _AUDIT_FEATURE_POOL if f not in _TEXT_COLS_ALL])}")
print(f"    text (all fams):  {len([f for f in _AUDIT_FEATURE_POOL if f in _TEXT_COLS_ALL])}")

_SHARED_ROOT = Path('cluster_data/shared')
_SHARED_ROOT.mkdir(parents=True, exist_ok=True)

with open(_SHARED_ROOT / 'audit_feature_pool.json', 'w') as f:
    json.dump({
        'n_features': len(_AUDIT_FEATURE_POOL),
        'features': _AUDIT_FEATURE_POOL,
        'forbidden': sorted(_FORBIDDEN_POOL),
        'text_families': {
            'structured_text_flags': _TEXT_COLS_STRUCT_FLAGS,
            'engineered_text':       _TEXT_COLS_ENGINEERED,
            'svd_text':              _TEXT_COLS_SVD,
            'embedding_text':        _TEXT_COLS_EMB,
        },
        'note': (
            'Unpruned pool for peer-group benchmarking. No redundancy '
            'pruning is applied: every interpretable feature is a candidate '
            'explanation for trial duration within a peer group.'
        ),
    }, f, indent=2)

print(f"  ✅ Saved {_SHARED_ROOT / 'audit_feature_pool.json'}")
print("=" * 70)


REFERENCE FEATURE POOL (unpruned)
  Audit feature pool: 107 features
    structural/bool:  91
    text (all fams):  16
  ✅ Saved cluster_data\shared\audit_feature_pool.json


In [17]:
# ============================================================================
# CELL 5.6: Drop raw list/dict columns (post-engineering cleanup)
# ============================================================================
# By the end of Cell 5.3, every raw list/dict column has been fully distilled
# into engineered features:
#   - locations        → num_sites, num_countries, us_sites, has_locations
#   - outcomes         → total_outcomes, outcome_complexity_total, has_*_outcome
#   - conditions       → num_conditions
#   - phases           → phase_set, phase_score, phase_count, has_phase*
#   - armGroups        → num_arm_groups
#   - eligibility text → elig_text_* (kept, mined in Cell 5.1b)
#
# These columns are the widest in the frame — each cell is a nested Python
# list of dicts, which pandas stores as PyObject pointers. Dropping them now
# saves ~300-400 MB before Cell 6 halves the row count.
#
# SAFETY: none of these columns appear in Cell 13's _REQUIRED_COLS list, and
# no downstream cell references them by name. Verified against the notebook.
#
# If you ever need the raw eligibility TEXT (eligibilityCriteria), it must
# NOT be in this list — Cell 13 writes a separate elig-text-features parquet
# that assumes the text has already been mined by Cell 5.1b. The raw text
# column itself is not required by any downstream cell after 5.1b runs.

_DROP_AFTER_ENGINEERING = [
    # Raw list-of-dicts
    'protocolSection.contactsLocationsModule.locations',
    'protocolSection.outcomesModule.primaryOutcomes',
    'protocolSection.outcomesModule.secondaryOutcomes',
    'protocolSection.outcomesModule.otherOutcomes',
    'protocolSection.armsInterventionsModule.armGroups',
    'protocolSection.conditionsModule.conditions',
    'protocolSection.designModule.phases',

    # Raw list-of-str / set intermediates created during engineering
    'outcome_types',
    'funding_sources',
    'phase_set',
]

_dropped = [c for c in _DROP_AFTER_ENGINEERING if c in df.columns]
_n_cols_before = df.shape[1]
df = df.drop(columns=_dropped)
_n_cols_after = df.shape[1]

gc.collect()

print("\n" + "=" * 70)
print("POST-ENGINEERING COLUMN CLEANUP")
print("=" * 70)
print(f"  Columns before:  {_n_cols_before}")
print(f"  Columns dropped: {len(_dropped)}")
print(f"  Columns after:   {_n_cols_after}")
print()
print("=" * 70)


POST-ENGINEERING COLUMN CLEANUP
  Columns before:  145
  Columns dropped: 10
  Columns after:   135



In [18]:
# ============================================================================
# CELL 6: Restrict df to modeled branches
# ============================================================================
_modelled = {'INTERVENTIONAL', 'OBSERVATIONAL'}
_all_st = df['study_type'].dropna().unique().tolist()
_excluded = [st for st in _all_st if st not in _modelled]
if _excluded:
    _n_excl = df[~df['study_type'].isin(_modelled)].shape[0]
    print(f"ℹ️ Excluding {_n_excl:,} trials from unmodelled study types:")
    for st in _excluded:
        print(f"   {st:<25} {(df['study_type'] == st).sum():>8,}")

df = df[df['study_type'].isin(_modelled)].copy()
df_interventional = df[df['study_type'] == 'INTERVENTIONAL'].copy()
df_observational  = df[df['study_type'] == 'OBSERVATIONAL'].copy()

print(f"Interventional: {len(df_interventional):,}")
print(f"Observational:  {len(df_observational):,}")
print(f"Total:          {len(df):,}")

ℹ️ Excluding 40 trials from unmodelled study types:
   EXPANDED_ACCESS                 40
Interventional: 136,579
Observational:  35,302
Total:          171,881


In [19]:
# ============================================================================
# CELL 7: Text Feature Inventory (from audit pool)
# ============================================================================

import json
from pathlib import Path

print("\n" + "=" * 74)
print("TEXT FEATURE INVENTORY")
print("=" * 74)

_SHARED_ROOT = Path('cluster_data/shared')
with open(_SHARED_ROOT / 'audit_feature_pool.json') as f:
    _pool = json.load(f)

_families = _pool['text_families']

TEXT_FEATURES_STRUCT_FLAGS = _families['structured_text_flags']
TEXT_FEATURES_ENGINEERED   = _families['engineered_text']
TEXT_FEATURES_SVD          = _families['svd_text']
TEXT_FEATURES_EMB          = _families['embedding_text']

TEXT_FEATURES = (TEXT_FEATURES_STRUCT_FLAGS + TEXT_FEATURES_ENGINEERED
                 + TEXT_FEATURES_SVD + TEXT_FEATURES_EMB)
TEXT_FEATURES = [f for f in TEXT_FEATURES if f in df_full.columns]

_has_flags      = len(TEXT_FEATURES_STRUCT_FLAGS) > 0
_has_engineered = len(TEXT_FEATURES_ENGINEERED) > 0
_has_svd        = len(TEXT_FEATURES_SVD) > 0
_has_emb        = len(TEXT_FEATURES_EMB) > 0

if _has_svd and _has_emb:              _representation = 'all'
elif _has_emb:                         _representation = 'embeddings'
elif _has_svd:                         _representation = 'tfidf_svd'
elif _has_engineered:                  _representation = 'engineered_only'
elif _has_flags:                       _representation = 'keyword_only'
else:                                  _representation = 'none'

print(f"  Text-representation: {_representation}")
print(f"  Families:")
print(f"    structured_text_flags:  {len(TEXT_FEATURES_STRUCT_FLAGS):>4}")
print(f"    engineered_text:        {len(TEXT_FEATURES_ENGINEERED):>4}")
print(f"    svd_text:               {len(TEXT_FEATURES_SVD):>4}")
print(f"    embedding_text:         {len(TEXT_FEATURES_EMB):>4}")
print(f"    TOTAL:                  {len(TEXT_FEATURES):>4}")

_TEMPORAL_REF_PATH = _SHARED_ROOT / 'temporal_reference.json'
_temporal_ref = None
if _TEMPORAL_REF_PATH.exists():
    with open(_TEMPORAL_REF_PATH) as f:
        _temporal_ref = json.load(f)

_TEXT_INVENTORY = {
    'schema_version':      2,
    'n_features':          len(TEXT_FEATURES),
    'text_representation': _representation,
    'feature_families': {
        'structured_text_flags': TEXT_FEATURES_STRUCT_FLAGS,
        'engineered_text':       TEXT_FEATURES_ENGINEERED,
        'svd_text':              TEXT_FEATURES_SVD,
        'embedding_text':        TEXT_FEATURES_EMB,
    },
    'feature_names':   TEXT_FEATURES,
    'temporal_reference': _temporal_ref,
    'notes': (
        'Text features available for peer-group attribution. No ablation '
        'is performed here; notebook 03 selects which families to attribute.'
    ),
}
with open(_SHARED_ROOT / 'text_feature_inventory.json', 'w') as f:
    json.dump(_TEXT_INVENTORY, f, indent=2)

print(f"\n  ✅ Saved {_SHARED_ROOT / 'text_feature_inventory.json'}")
print("=" * 74)


TEXT FEATURE INVENTORY
  Text-representation: engineered_only
  Families:
    structured_text_flags:     9
    engineered_text:           8
    svd_text:                  0
    embedding_text:            0
    TOTAL:                    17

  ✅ Saved cluster_data\shared\text_feature_inventory.json


In [20]:
# ============================================================================
# CELL 8: Strategy A partition helpers + eta² gate
# ============================================================================
# This cell defines the canonical Strategy A partition and evaluates it. The
# same helper is used by Cell 9.1 to ship the labels. Cell 9.2 asserts
# equality.
#
# The eta-squared diagnostic in this cell and the shipped partition in Cell
# 9.1 both call build_strategy_A_partition, so the gate measures exactly the
# partition that ships. Rows with num_sites == num_countries == 0 are
# bucketed as scope='unknown' rather than 'single-site' — missing location
# data is not the same thing as a single-site trial.

import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from clinical_trial_benchmarking.taxonomy import (
    OBSERVATIONAL_MODEL_MAP,
    OBSERVATIONAL_PERSPECTIVE_MAP,
)

print("\n" + "=" * 70)
print("STRATEGY A — CANONICAL PARTITION + ETA² GATE")
print("=" * 70)

ETA2_THRESHOLD = 0.03

def _normalize_obs_enum(raw, mapping):
    if raw is None:
        return 'other'
    try:
        if pd.isna(raw):
            return 'other'
    except (ValueError, TypeError):
        return 'other'
    s = str(raw).upper().strip().replace('-', '_').replace(' ', '_')
    return mapping.get(s, 'other')


def _build_stage_bucket(phase_score):
    """
    Ordinal phase_score -> stage bucket.
        0 -> 'unphased'
        1 -> 'exploratory'
        2 -> 'pivotal'
        3,4 -> 'confirmatory'
    """
    _ps = phase_score.fillna(0).astype(int)
    return pd.Series(
        np.select(
            [_ps == 0, _ps == 1, _ps == 2, _ps.isin([3, 4])],
            ['unphased', 'exploratory', 'pivotal', 'confirmatory'],
            default='unphased',
        ),
        index=phase_score.index,
    )


def _build_scope_bucket(num_sites, num_countries):
    """
    (sites, countries) -> scope bucket.

    FIX A2: rows with num_sites == 0 have missing location data, not a
    single-site trial. They are bucketed as 'unknown' so they do not
    contaminate the single-site bucket.
    """
    _s = num_sites.fillna(0).astype(int)
    _c = num_countries.fillna(0).astype(int)

    no_data = (_s == 0) & (_c == 0)
    single  = (_s <= 1) & (_c <= 1) & ~no_data
    regional = (_s <= 10) & (_c <= 3) & ~no_data & ~single

    return pd.Series(
        np.select(
            [no_data, single, regional],
            ['unknown', 'single-site', 'regional'],
            default='multinational',
        ),
        index=num_sites.index,
    )


def _build_observational_label(df_in):
    _M = 'protocolSection.designModule.designInfo.observationalModel'
    _P = 'protocolSection.designModule.designInfo.timePerspective'
    raw_m = df_in[_M] if _M in df_in.columns else pd.Series(['other'] * len(df_in), index=df_in.index)
    raw_p = df_in[_P] if _P in df_in.columns else pd.Series(['other'] * len(df_in), index=df_in.index)
    m = raw_m.apply(lambda v: _normalize_obs_enum(v, OBSERVATIONAL_MODEL_MAP))
    p = raw_p.apply(lambda v: _normalize_obs_enum(v, OBSERVATIONAL_PERSPECTIVE_MAP))
    return 'obs::' + m.astype(str) + '::' + p.astype(str)


def build_strategy_A_partition(df_in):
    """
    Canonical Strategy A partition label.

    Interventional -> "<stage>::<scope>"
    Observational  -> "obs::<model>::<perspective>"

    Both Cell 9 (this cell) and Cell 9.1 (shipped) call this. Cell 9.2
    asserts they agree.
    """
    is_interv = (df_in['study_type'] == 'INTERVENTIONAL')
    stage = _build_stage_bucket(df_in['phase_score'])
    scope = _build_scope_bucket(df_in['num_sites'], df_in['num_countries'])
    interv_label = stage.astype(str) + '::' + scope.astype(str)
    obs_label = _build_observational_label(df_in)
    return pd.Series(
        np.where(is_interv.values, interv_label.values, obs_label.values),
        index=df_in.index,
    )


def _eta_squared_on_log_target(df_b, bucket_col, target_col):
    groups = [df_b.loc[df_b[bucket_col] == b, target_col].values
              for b in df_b[bucket_col].dropna().unique()]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        return 0.0, None, None
    all_vals = np.concatenate(groups)
    grand = all_vals.mean()
    ss_tot = ((all_vals - grand) ** 2).sum()
    ss_bet = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    eta2 = ss_bet / ss_tot if ss_tot > 0 else 0.0
    F, p = f_oneway(*groups)
    return float(eta2), float(F), float(p)


# Diagnostic run on the CANONICAL partition.
df_interventional['_prov_bucket'] = build_strategy_A_partition(df_interventional)
df_observational['_prov_bucket']  = build_strategy_A_partition(df_observational)

for _branch, _df in [('INTERVENTIONAL', df_interventional),
                     ('OBSERVATIONAL', df_observational)]:
    eta2, F, p = _eta_squared_on_log_target(_df, '_prov_bucket', 'duration_days_log')
    print(f"\n{_branch}:")
    print(f"  Eta²: {eta2:.6f}   ANOVA F = {F:,.1f}   p = {p:.2e}"
          if F is not None else f"  Eta²: {eta2:.4f}")
    print(f"  {'✅ PASSED' if eta2 >= ETA2_THRESHOLD else '❌ FAILED'}")

ETA2_INTERVENTIONAL = float(_eta_squared_on_log_target(
    df_interventional, '_prov_bucket', 'duration_days_log')[0])
ETA2_OBSERVATIONAL = float(_eta_squared_on_log_target(
    df_observational, '_prov_bucket', 'duration_days_log')[0])

PARTITION_VIABLE = (ETA2_INTERVENTIONAL >= ETA2_THRESHOLD or
                    ETA2_OBSERVATIONAL >= ETA2_THRESHOLD)

df_interventional = df_interventional.drop(columns=['_prov_bucket'], errors='ignore')
df_observational = df_observational.drop(columns=['_prov_bucket'], errors='ignore')

print(f"\nPartition viable: {PARTITION_VIABLE}")
if not PARTITION_VIABLE:
    # Note: this notebook continues even if the gate fails. The downstream
    # decision (03) is the final authority. Cell 9.2 will still assert
    # diagnostic == shipped.
    print("  ⚠️ Partition failed eta² gate. Downstream cells will run, but the")
    print("     final deployment decision is expected to be KEEP_GLOBAL.")

print("=" * 70)


STRATEGY A — CANONICAL PARTITION + ETA² GATE

INTERVENTIONAL:
  Eta²: 0.046897   ANOVA F = 448.0   p = 0.00e+00
  ✅ PASSED

OBSERVATIONAL:
  Eta²: 0.046919   ANOVA F = 64.3   p = 0.00e+00
  ✅ PASSED

Partition viable: True


In [21]:
# ============================================================================
# CELL 9: Ship the canonical Strategy A partition
# ============================================================================
# Uses the helpers from Cell 9. The partition shipped here is byte-identical
# to the one Cell 9's eta² gate measured; Cell 9.2 asserts that.

print("\n" + "=" * 70)
print("STRATEGY A — SHIPPED LABELS")
print("=" * 70)

N_MIN = 2000

# Canonical regime_bucket
df['regime_bucket'] = build_strategy_A_partition(df).astype(str)

# Individual axis columns for downstream use.
_is_interv = (df['study_type'] == 'INTERVENTIONAL').values

_stage = _build_stage_bucket(df['phase_score'])
_scope = _build_scope_bucket(df['num_sites'], df['num_countries'])

df['strategy_A_stage'] = np.where(_is_interv, _stage.values, 'unphased')
df['strategy_A_scope'] = np.where(_is_interv, _scope.values, 'unphased')

_M = 'protocolSection.designModule.designInfo.observationalModel'
_P = 'protocolSection.designModule.designInfo.timePerspective'
raw_m = df[_M] if _M in df.columns else pd.Series(['other'] * len(df), index=df.index)
raw_p = df[_P] if _P in df.columns else pd.Series(['other'] * len(df), index=df.index)

_m_norm = raw_m.apply(lambda v: _normalize_obs_enum(v, OBSERVATIONAL_MODEL_MAP))
_p_norm = raw_p.apply(lambda v: _normalize_obs_enum(v, OBSERVATIONAL_PERSPECTIVE_MAP))

df['strategy_A_obs_model'] = np.where(_is_interv, 'unphased', _m_norm.values)
df['strategy_A_obs_perspective'] = np.where(_is_interv, 'unphased', _p_norm.values)

# Retain therapeutic_area for the deterministic prior in Strategy D.
_ta_cols = [c for c in THERAPEUTIC_AREA_COLUMNS if c in df.columns]
if _ta_cols:
    df['therapeutic_area'] = np.select(
        [df[c].astype(bool).values for c in _ta_cols],
        [c.replace('is_', '') for c in _ta_cols],
        default='other',
    )
else:
    df['therapeutic_area'] = 'other'

# Refresh branches.
df_interventional = df[df['study_type'] == 'INTERVENTIONAL'].copy()
df_observational  = df[df['study_type'] == 'OBSERVATIONAL'].copy()

# Distribution diagnostics.
for _branch in ['INTERVENTIONAL', 'OBSERVATIONAL']:
    _df_b = df[df['study_type'] == _branch]
    dist = _df_b['regime_bucket'].value_counts().sort_values(ascending=False)
    print(f"\n{_branch}  (n={len(_df_b):,}):")
    print(f"  {'bucket':<45} {'n':>9} {'pct':>7}  {'≥N_min':>8}")
    for bucket, n in dist.items():
        meets = "✅" if n >= N_MIN else "❌"
        print(f"  {bucket:<45} {n:>9,} {100*n/len(_df_b):>6.1f}%  {meets:>8}")
    _below = (dist < N_MIN).sum()
    print(f"  Buckets below N_min: {_below} / {len(dist)}")

print("=" * 70)


STRATEGY A — SHIPPED LABELS

INTERVENTIONAL  (n=136,579):
  bucket                                                n     pct    ≥N_min
  unphased::single-site                            42,265   30.9%         ✅
  confirmatory::single-site                        17,458   12.8%         ✅
  pivotal::single-site                             12,465    9.1%         ✅
  exploratory::single-site                          9,714    7.1%         ✅
  unphased::regional                                8,747    6.4%         ✅
  unphased::unknown                                 8,452    6.2%         ✅
  confirmatory::multinational                       7,586    5.6%         ✅
  pivotal::regional                                 5,443    4.0%         ✅
  confirmatory::regional                            5,354    3.9%         ✅
  pivotal::multinational                            4,721    3.5%         ✅
  confirmatory::unknown                             4,363    3.2%         ✅
  exploratory::regional      

In [22]:
# ============================================================================
# CELL 10: Assert diagnostic == shipped
# ============================================================================
# If this fails, the eta² gate measured a partition other than the shipped
# one, invalidating the gate.

print("\n" + "=" * 70)
print("PARTITION CONSISTENCY ASSERTION")
print("=" * 70)

_diag = build_strategy_A_partition(df).astype(str)
_shipped = df['regime_bucket'].astype(str)

_mismatch = (_diag != _shipped)
_n_mismatch = int(_mismatch.sum())

if _n_mismatch == 0:
    print(f"  ✅ Diagnostic partition == shipped partition on all {len(df):,} rows.")
else:
    _examples = pd.DataFrame({
        'diagnostic': _diag[_mismatch].head(5).tolist(),
        'shipped':    _shipped[_mismatch].head(5).tolist(),
    })
    print(f"  ❌ Mismatch on {_n_mismatch:,} / {len(df):,} rows.")
    print(_examples.to_string(index=False))
    raise RuntimeError(
        f"❌ Eta² gate measured a partition that differs from the shipped "
        f"one on {_n_mismatch:,} rows. The gate is invalid."
    )

print(f"  Eta² — INTERVENTIONAL: {ETA2_INTERVENTIONAL:.4f}")
print(f"  Eta² — OBSERVATIONAL:  {ETA2_OBSERVATIONAL:.4f}")
print("=" * 70)


PARTITION CONSISTENCY ASSERTION
  ✅ Diagnostic partition == shipped partition on all 171,881 rows.
  Eta² — INTERVENTIONAL: 0.0469
  Eta² — OBSERVATIONAL:  0.0469


In [23]:
# ============================================================================
# CELL 11: EXPORT SHARED BUNDLE (peer-group version)
# ============================================================================
# Writes the reference corpus, canonical split, peer-group assignments,
# and all downstream artifacts needed by the audit notebook.

from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

_SHARED_ROOT = Path('cluster_data/shared')
_SHARED_ROOT.mkdir(parents=True, exist_ok=True)

_TRIAL_ID_COL = 'protocolSection.identificationModule.nctId'

# ---------------------------------------------------------------------------
# 13a  Canonical split
# ---------------------------------------------------------------------------
_split_df = pd.DataFrame({
    'trial_id': df[_TRIAL_ID_COL].astype(str),
    'split':    df['_split'].astype(str),
})
if not _split_df['trial_id'].is_unique:
    raise RuntimeError(
        f"❌ {int(_split_df['trial_id'].duplicated().sum()):,} duplicate trial_ids"
    )
_split_path = _SHARED_ROOT / 'canonical_split.parquet'
_split_df.to_parquet(_split_path, index=False)
print(f"✅ {_split_path}")
print(f"   {int((_split_df['split']=='train').sum()):,} train / "
      f"{int((_split_df['split']=='test').sum()):,} test")

# ---------------------------------------------------------------------------
# 13b  Enriched dataframe (reference corpus)
# ---------------------------------------------------------------------------
_enriched_path = _SHARED_ROOT / 'trial_reference_corpus.parquet'

if df['_split'].isna().any():
    raise RuntimeError(f"❌ {int(df['_split'].isna().sum()):,} rows with no split")
if df['regime_bucket'].isna().any():
    raise RuntimeError(f"❌ {int(df['regime_bucket'].isna().sum()):,} rows with no peer group")
if not df[_TRIAL_ID_COL].astype(str).is_unique:
    raise RuntimeError(
        f"❌ {int(df[_TRIAL_ID_COL].astype(str).duplicated().sum()):,} duplicate trial IDs"
    )

df.to_parquet(_enriched_path, index=False)
print(f"\n✅ {_enriched_path}")
print(f"   Rows:    {len(df):,}")
print(f"   Columns: {df.shape[1]}")
print(f"   Size:    {_enriched_path.stat().st_size / (1024**2):.1f} MB")

# ---------------------------------------------------------------------------
# 13c  Peer-group assignments (single artifact)
# ---------------------------------------------------------------------------
_assign = pd.DataFrame({
    'trial_id':         df[_TRIAL_ID_COL].astype(str),
    'peer_group':       df['regime_bucket'].astype(str),
    'stage':            df['strategy_A_stage'].astype(str),
    'scope':            df['strategy_A_scope'].astype(str),
    'obs_model':        df['strategy_A_obs_model'].astype(str),
    'obs_perspective':  df['strategy_A_obs_perspective'].astype(str),
    'study_type':       df['study_type'].astype(str),
    'duration_days':    df['duration_days'].astype(float),
})
_assign_path = _SHARED_ROOT / 'peer_group_assignments.parquet'
_assign.to_parquet(_assign_path, index=False)
print(f"\n✅ {_assign_path}")
print(f"   {_assign['peer_group'].nunique()} distinct peer groups")

# ---------------------------------------------------------------------------
# 13d  Eligibility text features (unchanged)
# ---------------------------------------------------------------------------
_text_cols = sorted(c for c in df.columns if c.startswith('elig_text_'))
if _text_cols:
    _text_df = df[[_TRIAL_ID_COL] + _text_cols].rename(
        columns={_TRIAL_ID_COL: 'trial_id'}
    )
    _text_df['trial_id'] = _text_df['trial_id'].astype(str)
    _text_path = _SHARED_ROOT / 'eligibility_text_features.parquet'
    _text_df.to_parquet(_text_path, index=False)
    print(f"\n✅ {_text_path}")
    print(f"   Rows: {len(_text_df):,}   Cols: {len(_text_cols)} text + trial_id")

# ---------------------------------------------------------------------------
# 13e  Summary manifest
# ---------------------------------------------------------------------------
_manifest = {
    'schema_version': 1,
    'generated_by':   '02_clinical_trial_peer_grouping',
    'generated_at':   datetime.now(timezone.utc).isoformat(),

    'trial_id_col':   _TRIAL_ID_COL,
    'n_trials':       int(len(df)),
    'n_peer_groups':  int(df['regime_bucket'].nunique()),

    'peer_axis': {
        'method': 'deterministic_branch_specific_slice',
        'interventional': 'stage × scope',
        'observational':  'obs::model::perspective',
        'notes': (
            'Peer groups are deterministic. Same trial design features -> '
            'same peer group, always. No clustering, no randomness.'
        ),
    },

    'paths': {
        'reference_corpus':       str(_enriched_path),
        'canonical_split':        str(_split_path),
        'peer_assignments':       str(_assign_path),
        'peer_distributions':     str(_SHARED_ROOT / 'peer_distributions.parquet'),
        'peer_feature_profiles':  str(_SHARED_ROOT / 'peer_feature_profiles.json'),
        'peer_risk_factors':      str(_SHARED_ROOT / 'peer_risk_factors.json'),
        'audit_feature_pool':     str(_SHARED_ROOT / 'audit_feature_pool.json'),
        'text_feature_inventory': str(_SHARED_ROOT / 'text_feature_inventory.json'),
        'temporal_reference':     str(_SHARED_ROOT / 'temporal_reference.json'),
        'eligibility_text':       str(_SHARED_ROOT / 'eligibility_text_features.parquet'),
    },
}
_manifest_path = _SHARED_ROOT / 'strategy_manifest.json'
with open(_manifest_path, 'w') as f:
    json.dump(_manifest, f, indent=2)

print(f"\n✅ {_manifest_path}")
print("=" * 70)

✅ cluster_data\shared\canonical_split.parquet
   137,454 train / 34,427 test

✅ cluster_data\shared\trial_reference_corpus.parquet
   Rows:    171,881
   Columns: 141
   Size:    117.4 MB

✅ cluster_data\shared\peer_group_assignments.parquet
   44 distinct peer groups

✅ cluster_data\shared\eligibility_text_features.parquet
   Rows: 171,881   Cols: 8 text + trial_id

✅ cluster_data\shared\strategy_manifest.json


In [24]:
# ============================================================================
# CELL 12: PEER DISTRIBUTION TABLE
# ============================================================================
# For every peer group (regime_bucket), compute the empirical distribution of
# actual trial duration. This is the reference table the audit looks up.
#
# Also stores per-group feature medians, so the audit can describe a peer
# group without re-querying the full corpus.

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("\n" + "=" * 70)
print("PEER DISTRIBUTION TABLE")
print("=" * 70)

_SHARED_ROOT = Path('cluster_data/shared')

# regime_bucket was assigned in Cell 9.1.
if 'regime_bucket' not in df.columns:
    raise RuntimeError("❌ regime_bucket not found. Cell 9.1 must run first.")

_PERCS = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

_rows = []
_feature_profiles = {}

for _bucket, _sub in df.groupby('regime_bucket'):
    _d = _sub['duration_days'].dropna()
    if len(_d) == 0:
        continue

    _row = {
        'regime_bucket': _bucket,
        'n_trials':      int(len(_sub)),
        'n_with_duration': int(len(_d)),
        'mean_days':     float(_d.mean()),
        'std_days':      float(_d.std()),
        'min_days':      float(_d.min()),
        'max_days':      float(_d.max()),
    }
    for _p in _PERCS:
        _row[f'p{int(_p*100):02d}_days'] = float(_d.quantile(_p))

    # Category mix
    if 'duration_category_business' in _sub.columns:
        _mix = _sub['duration_category_business'].value_counts(normalize=True)
        _row['pct_small']  = float(_mix.get('Small (<1yr)', 0.0))
        _row['pct_medium'] = float(_mix.get('Medium (1-3yr)', 0.0))
        _row['pct_large']  = float(_mix.get('Large (>3yr)', 0.0))

    _rows.append(_row)

    # Per-group feature medians (for describing the peer group in the audit)
    _numeric_feats = [f for f in _AUDIT_FEATURE_POOL
                      if f in df.columns
                      and pd.api.types.is_numeric_dtype(df[f])]
    _feature_profiles[_bucket] = {
        'n': int(len(_sub)),
        'feature_medians': {
            f: float(_sub[f].median()) for f in _numeric_feats
        },
    }

_peer_dist = pd.DataFrame(_rows).set_index('regime_bucket').sort_index()

# Minimum-n gate — flag groups that are too small for meaningful benchmarking
_MIN_PEER_N = 30
_peer_dist['reliable'] = _peer_dist['n_with_duration'] >= _MIN_PEER_N

_out_path = _SHARED_ROOT / 'peer_distributions.parquet'
_peer_dist.reset_index().to_parquet(_out_path, index=False)

_prof_path = _SHARED_ROOT / 'peer_feature_profiles.json'
with open(_prof_path, 'w') as f:
    json.dump(_feature_profiles, f, indent=2)

print(f"  Peer groups:               {len(_peer_dist)}")
print(f"  Reliable (n>={_MIN_PEER_N}):       {int(_peer_dist['reliable'].sum())}")
print(f"  Unreliable (<{_MIN_PEER_N}):       {int((~_peer_dist['reliable']).sum())}")
print()
print(f"  Sample peer groups (median days):")
for _b in _peer_dist.head(8).index:
    _r = _peer_dist.loc[_b]
    print(f"    {_b:<42} n={int(_r['n_with_duration']):>6,}  "
          f"median={_r['p50_days']:>7.0f}d")

print(f"\n  ✅ Saved {_out_path}")
print(f"  ✅ Saved {_prof_path}")
print("=" * 70)


PEER DISTRIBUTION TABLE
  Peer groups:               44
  Reliable (n>=30):       39
  Unreliable (<30):       5

  Sample peer groups (median days):
    confirmatory::multinational                n= 7,586  median=    823d
    confirmatory::regional                     n= 5,354  median=    851d
    confirmatory::single-site                  n=17,458  median=    820d
    confirmatory::unknown                      n= 4,363  median=    669d
    exploratory::multinational                 n=   447  median=    943d
    exploratory::regional                      n= 3,358  median=    852d
    exploratory::single-site                   n= 9,714  median=    579d
    exploratory::unknown                       n= 1,632  median=    365d

  ✅ Saved cluster_data\shared\peer_distributions.parquet
  ✅ Saved cluster_data\shared\peer_feature_profiles.json


In [25]:
# ============================================================================
# CELL 13: PER-PEER-GROUP ATTRIBUTION (SHAP risk factors)
# ============================================================================
# Fit ONE global attribution model on the train fold. Then compute SHAP
# values PER PEER GROUP on the full corpus to learn which features push
# duration up or down WITHIN each peer group.
#
# This is attribution, not prediction. The model is a lens, not a product.
#
# R² is reported on BOTH train and test folds. The test fold is the
# held-out canonical split. A large train-test gap means the attribution
# lens is overfitting and the SHAP rankings below should be treated with
# reduced confidence. The gap is persisted to peer_risk_factors.json so
# downstream notebooks can display it.
#
# FIX (categorization): every attribution record now carries:
#   - 'category'         : one of design_lever | era_cohort | sponsor |
#                          therapeutic_area | text_context | other
#   - 'category_label'   : short display label ('design', 'era', ...)
#   - 'direction_gloss'  : a sentence describing what the sign means,
#                          or '' if no gloss is registered
#
# These fields are display aids. They do not affect the model, the SHAP
# values, or the ranking. Consumers that ignore them see byte-identical
# behavior. The schema_version stays at 2 because these are additive.
# ============================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# Categorization lives in the package so notebook 03 can import the same
# definitions without duplicating them.
from clinical_trial_benchmarking.taxonomy import categorize_feature
from clinical_trial_benchmarking.audit import (
    category_display_label,
    gloss as _direction_gloss,
)

print("\n" + "=" * 70)
print("PER-PEER-GROUP ATTRIBUTION (SHAP)")
print("=" * 70)

_SHARED_ROOT = Path('cluster_data/shared')

# ---------------------------------------------------------------------------
# 13.2a  Encode the audit feature pool (fit on train fold only)
# ---------------------------------------------------------------------------
_AUDIT_FEATURES = [f for f in _AUDIT_FEATURE_POOL if f in df.columns]

def _encode_for_attribution(X_train_raw, X_all_raw):
    """
    Fit one-hot encoding + median imputation on X_train_raw, apply to both.
    X_all_raw is reindexed to X_train_raw's encoded columns.
    """
    X_train_raw = X_train_raw.copy()
    X_all_raw   = X_all_raw.copy()
    for c in X_train_raw.columns:
        if c not in X_all_raw.columns:
            X_all_raw[c] = 0
    X_all_raw = X_all_raw[X_train_raw.columns]

    cat_cols = X_train_raw.select_dtypes(include=['object']).columns.tolist()
    num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()

    for c in num_cols:
        med = X_train_raw[c].median()
        if pd.isna(med):
            med = 0.0
        X_train_raw[c] = X_train_raw[c].fillna(med)
        X_all_raw[c]   = X_all_raw[c].fillna(med)
    for c in cat_cols:
        X_train_raw[c] = X_train_raw[c].fillna('UNKNOWN').astype(str)
        X_all_raw[c]   = X_all_raw[c].fillna('UNKNOWN').astype(str)

    if cat_cols:
        X_tr  = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True)
        X_all = pd.get_dummies(X_all_raw,   columns=cat_cols, drop_first=True)
    else:
        X_tr, X_all = X_train_raw, X_all_raw

    X_all = X_all.reindex(columns=X_tr.columns, fill_value=0)
    return X_tr, X_all

_train_df = df[df['_split'] == 'train']
_test_df  = df[df['_split'] == 'test']

# Fit encoder on train only, then apply to train and full corpus.
X_tr, X_all = _encode_for_attribution(
    _train_df[_AUDIT_FEATURES],
    df[_AUDIT_FEATURES],
)

y_tr  = _train_df['duration_days_log'].values
y_all = df['duration_days_log'].values

# ---------------------------------------------------------------------------
# 13.2b  Fit the global attribution model + report train AND test R²
# ---------------------------------------------------------------------------
_attr_model = XGBRegressor(
    n_estimators=300, learning_rate=0.04, max_depth=6,
    random_state=42, n_jobs=-1,
)
_attr_model.fit(X_tr, y_tr)

_r2_train = float(r2_score(y_tr, _attr_model.predict(X_tr)))

# Test fold — indexed by row label so we can look up in X_all safely.
_X_te = X_all.loc[_test_df.index]
_y_te = _test_df['duration_days_log'].values
_r2_test = float(r2_score(_y_te, _attr_model.predict(_X_te)))
_r2_gap = _r2_train - _r2_test

print(f"  Attribution model: XGBRegressor")
print(f"    R² (train): {_r2_train:.4f}")
print(f"    R² (test):  {_r2_test:.4f}")
print(f"    R² gap:     {_r2_gap:+.4f}")

if _r2_gap > 0.10:
    print(f"  ⚠️  Train-test R² gap > 0.10 — the attribution lens is "
          f"overfitting.")
    print(f"      SHAP rankings below should be treated with reduced "
          f"confidence.")
elif _r2_gap < -0.02:
    print(f"  ℹ️  Test R² exceeds train R² — the model is not overfitting. "
          f"This usually indicates the test fold is easier to fit.")

print(f"  Encoded feature schema: {X_tr.shape[1]} columns")

# ---------------------------------------------------------------------------
# 13.2c  Compute SHAP per peer group
#
# Each attribution record is enriched with category metadata and a
# directional gloss. The gloss is looked up per (feature, direction)
# because the SIGN of a feature's association can flip between peer
# groups — e.g. more sites accelerates pivotal trials but decelerates
# single-site trials.
# ---------------------------------------------------------------------------
_explainer = shap.TreeExplainer(_attr_model)
_top_features_per_peer = {}
_n_per_group = {}
_category_counts = {}   # for the diagnostic print below

for _bucket, _sub_idx in df.groupby('regime_bucket').groups.items():
    if len(_sub_idx) < 30:
        # Too small for stable attributions
        continue
    _X_sub = X_all.loc[_sub_idx]
    _sv = _explainer.shap_values(_X_sub)
    if isinstance(_sv, list):
        _sv = _sv[0]
    _sv = np.asarray(_sv)
    if _sv.ndim == 3:
        _sv = _sv.mean(axis=-1)

    _mean_abs    = np.abs(_sv).mean(axis=0)
    _mean_signed = _sv.mean(axis=0)

    _imp = pd.DataFrame({
        'feature':          X_all.columns,
        'mean_abs_shap':    _mean_abs,
        'mean_signed_shap': _mean_signed,
    }).sort_values('mean_abs_shap', ascending=False).head(20)

    _imp['direction'] = np.where(_imp['mean_signed_shap'] >= 0, '+', '-')

    # Enrich each record with categorization + gloss.
    _records = []
    for _, _row in _imp.iterrows():
        _feat = _row['feature']
        _dir  = _row['direction']
        _cat  = categorize_feature(_feat)

        _records.append({
            'feature':          _feat,
            'mean_abs_shap':    float(_row['mean_abs_shap']),
            'mean_signed_shap': float(_row['mean_signed_shap']),
            'direction':        _dir,
            'category':         _cat,
            'category_label':   category_display_label(_cat),
            'direction_gloss':  _direction_gloss(_feat, _dir),
        })

    _top_features_per_peer[_bucket] = _records
    _n_per_group[_bucket] = int(len(_sub_idx))

    # Track category coverage for the diagnostic below. We count how
    # many peer groups have at least one design_lever in the top-20 —
    # this is the metric that determines whether the audit report will
    # be able to surface an actionable block.
    _levers_in_top = sum(
        1 for r in _records if r['category'] == 'design_lever'
    )
    _category_counts.setdefault('design_lever', 0)
    if _levers_in_top > 0:
        _category_counts['design_lever'] += 1

    print(f"    {_bucket:<42} n={len(_sub_idx):>6,}  "
          f"top: {_imp.iloc[0]['feature']} ({_imp.iloc[0]['direction']})  "
          f"[levers in top-20: {_levers_in_top}]")

# ---------------------------------------------------------------------------
# 13.2d  Persist
# ---------------------------------------------------------------------------
_out_path = _SHARED_ROOT / 'peer_risk_factors.json'
with open(_out_path, 'w') as f:
    json.dump({
        'schema_version':       2,   # unchanged: categorization is additive
        'model_family':         'XGBRegressor',
        'model_r2_train':       _r2_train,
        'model_r2_test':        _r2_test,
        'model_r2_gap':         _r2_gap,
        'n_encoded_features':   int(X_tr.shape[1]),
        'n_peer_groups_scored': len(_top_features_per_peer),
        'n_peers_per_group':    _n_per_group,
        'risk_factors':         _top_features_per_peer,
        'notes': (
            'SHAP attributions computed per peer group on the full corpus. '
            'Positive direction means the feature is associated with longer '
            'durations within that peer group; negative means shorter. '
            'R² is reported on both the train fold (used to fit the model) '
            'and the held-out test fold (used only for the diagnostic). '
            'A large train-test gap means the attributions carry '
            'overfitting risk and should be interpreted with caution. '
            'Each record carries a category (design_lever, era_cohort, '
            'sponsor, therapeutic_area, text_context, other) so downstream '
            'consumers can distinguish design choices from contextual '
            'signals; category assignment is a display aid, not a gate.'
        ),
    }, f, indent=2)

print(f"\n  ✅ Saved {_out_path}")
print(f"  Peer groups with attributions: {len(_top_features_per_peer)}")

# ---------------------------------------------------------------------------
# 13.2e  Category coverage diagnostic
#
# The audit report gains an ACTIONABLE FEATURES block whenever a peer
# group's top-20 contains at least one design lever. If many peer
# groups lack one, that is itself a finding worth surfacing here — it
# means the tool's actionable layer has limited reach for those groups.
# ---------------------------------------------------------------------------
_n_with_levers = _category_counts.get('design_lever', 0)
_n_total = len(_top_features_per_peer)
_pct = 100.0 * _n_with_levers / _n_total if _n_total else 0.0

print(f"\n  Category coverage diagnostic:")
print(f"    Peer groups with at least one design lever in top-20: "
      f"{_n_with_levers} / {_n_total}  ({_pct:.1f}%)")

if _n_with_levers < _n_total:
    print(f"    ⚠️  {_n_total - _n_with_levers} peer group(s) have no design "
          f"lever in the top-20.")
    print(f"        For those groups, the audit's ACTIONABLE FEATURES block "
          f"will be empty.")
    print(f"        This is not an error — it means the position of a trial "
          f"in those groups")
    print(f"        is driven primarily by era, sponsor, or therapeutic area.")

print("=" * 70)


PER-PEER-GROUP ATTRIBUTION (SHAP)
  Attribution model: XGBRegressor
    R² (train): 0.4134
    R² (test):  0.3819
    R² gap:     +0.0316
  Encoded feature schema: 107 columns
    confirmatory::multinational                n= 7,586  top: funding_industry (-)  [levers in top-20: 12]
    confirmatory::regional                     n= 5,354  top: funding_industry (-)  [levers in top-20: 11]
    confirmatory::single-site                  n=17,458  top: years_since_registration (+)  [levers in top-20: 11]
    confirmatory::unknown                      n= 4,363  top: years_since_registration (+)  [levers in top-20: 12]
    exploratory::multinational                 n=   447  top: funding_industry (-)  [levers in top-20: 12]
    exploratory::regional                      n= 3,358  top: funding_industry (-)  [levers in top-20: 11]
    exploratory::single-site                   n= 9,714  top: funding_industry (-)  [levers in top-20: 12]
    exploratory::unknown                       n= 1,632  t

In [26]:
# ============================================================================
# CELL 14: PEER-GROUP SUMMARY REPORT
# ============================================================================
# Extended to print both train and test R² for the attribution model, plus
# a drift warning when the train-test gap exceeds the 0.10 threshold.
# ============================================================================

from pathlib import Path
import json
import pandas as pd

_SHARED_ROOT = Path('cluster_data/shared')

print("\n" + "=" * 78)
print("PEER-GROUP SUMMARY")
print("=" * 78)

# Load artifacts
with open(_SHARED_ROOT / 'strategy_manifest.json') as f:
    _manifest = json.load(f)
with open(_SHARED_ROOT / 'peer_risk_factors.json') as f:
    _risk = json.load(f)

_peer_dist = pd.read_parquet(_SHARED_ROOT / 'peer_distributions.parquet')

print(f"\n📊 REFERENCE CORPUS")
print("-" * 78)
print(f"   Total trials:       {_manifest['n_trials']:,}")
print(f"   Peer groups:        {_manifest['n_peer_groups']}")

# Largest / smallest
_top5 = _peer_dist.sort_values('n_with_duration', ascending=False).head(5)
_bot5 = _peer_dist.sort_values('n_with_duration', ascending=True).head(5)

print(f"\n📈 LARGEST PEER GROUPS")
print("-" * 78)
for _, _r in _top5.iterrows():
    print(f"   {_r['regime_bucket']:<42} n={int(_r['n_with_duration']):>6,}  "
          f"median={_r['p50_days']:>7.0f}d")

print(f"\n📉 SMALLEST PEER GROUPS")
print("-" * 78)
for _, _r in _bot5.iterrows():
    _flag = "" if _r['reliable'] else "  ⚠️"
    print(f"   {_r['regime_bucket']:<42} n={int(_r['n_with_duration']):>6,}  "
          f"median={_r['p50_days']:>7.0f}d{_flag}")

# Distribution summary
print(f"\n📐 DURATION DISTRIBUTION (all trials)")
print("-" * 78)
print(f"   p10:    {_peer_dist['p10_days'].min():.0f}  (min across groups)")
print(f"   median: {_peer_dist['p50_days'].median():.0f}  (median of medians)")
print(f"   p90:    {_peer_dist['p90_days'].max():.0f}  (max across groups)")

# Reliability
_n_unreliable = int((~_peer_dist['reliable']).sum())
if _n_unreliable:
    print(f"\n⚠️  {_n_unreliable} peer group(s) have fewer than 30 trials.")
    print(f"    These will fall back to a coarser grouping during audit.")

# Risk-factor coverage — now with train/test R²
print(f"\n🎯 RISK-FACTOR ATTRIBUTION")
print("-" * 78)
print(f"   Peer groups with SHAP attributions: {_risk['n_peer_groups_scored']}")
print(f"   Attribution model R² (train):       {_risk['model_r2_train']:.4f}")

# FIX (Priority #2): report test R² and gap when available. Backwards-
# compatible with schema_version=1 files that lack these fields.
_r2_test = _risk.get('model_r2_test')
_r2_gap  = _risk.get('model_r2_gap')

if _r2_test is not None and _r2_gap is not None:
    print(f"   Attribution model R² (test):        {_r2_test:.4f}")
    print(f"   Attribution model R² gap:           {_r2_gap:+.4f}")

    if _r2_gap > 0.10:
        print(f"\n   ⚠️  Train-test R² gap exceeds 0.10 — the attribution "
              f"model is overfitting.")
        print(f"       SHAP rankings in the audit should be read as "
              f"directional, not precise.")
    elif _r2_gap < -0.02:
        print(f"\n   ℹ️  Test R² exceeds train R² — no evidence of "
              f"overfitting.")
    else:
        print(f"\n   ✅ Train-test R² gap is within tolerance (< 0.10).")
else:
    print(f"   ⚠️  R²(test) missing from peer_risk_factors.json — "
          f"re-run Cell 13 to populate it.")

print("\n" + "=" * 78)


PEER-GROUP SUMMARY

📊 REFERENCE CORPUS
------------------------------------------------------------------------------
   Total trials:       171,881
   Peer groups:        44

📈 LARGEST PEER GROUPS
------------------------------------------------------------------------------
   unphased::single-site                      n=42,265  median=    790d
   confirmatory::single-site                  n=17,458  median=    820d
   obs::cohort::prospective                   n=13,752  median=   1033d
   pivotal::single-site                       n=12,465  median=   1065d
   exploratory::single-site                   n= 9,714  median=    579d

📉 SMALLEST PEER GROUPS
------------------------------------------------------------------------------
   obs::case-crossover::other                 n=     7  median=   1125d  ⚠️
   obs::ecologic::other                       n=    14  median=    410d  ⚠️
   obs::family-based::retrospective           n=    16  median=   1840d  ⚠️
   obs::family-based::other    

In [27]:
# ============================================================================
# CELL 15: PIPELINE VALIDATION — PEER-GROUP CONTRACT
# ============================================================================

from pathlib import Path
import pandas as pd
import json
import numpy as np

_SHARED_ROOT = Path('cluster_data/shared')

print("\n" + "=" * 70)
print("PIPELINE VALIDATION — PEER-GROUP CONTRACT")
print("=" * 70)

# ---------------------------------------------------------------------------
# 15a  Required artifacts
# ---------------------------------------------------------------------------
_REQUIRED = [
    'trial_reference_corpus.parquet',
    'canonical_split.parquet',
    'peer_group_assignments.parquet',
    'peer_distributions.parquet',
    'peer_feature_profiles.json',
    'peer_risk_factors.json',
    'audit_feature_pool.json',
    'text_feature_inventory.json',
    'temporal_reference.json',
    'eligibility_text_features.parquet',
    'strategy_manifest.json',
]
for _f in _REQUIRED:
    _p = _SHARED_ROOT / _f
    assert _p.exists(), f"❌ Missing artifact: {_p}"
    print(f"✅ {_p}")

# ---------------------------------------------------------------------------
# 15b  Trial universe consistency
# ---------------------------------------------------------------------------
_assign = pd.read_parquet(_SHARED_ROOT / 'peer_group_assignments.parquet')
_split  = pd.read_parquet(_SHARED_ROOT / 'canonical_split.parquet')
_text   = pd.read_parquet(_SHARED_ROOT / 'eligibility_text_features.parquet',
                          columns=['trial_id'])

_a_ids = set(_assign['trial_id'].astype(str))
_s_ids = set(_split['trial_id'].astype(str))
_t_ids = set(_text['trial_id'].astype(str))
assert _a_ids == _s_ids == _t_ids, (
    f"❌ Trial universes differ: assignments={len(_a_ids)}, "
    f"split={len(_s_ids)}, text={len(_t_ids)}"
)
print(f"\n✅ Trial universe consistent across all artifacts: {len(_a_ids):,}")

# ---------------------------------------------------------------------------
# 15c  Every trial has a peer group; every peer group has a distribution
# ---------------------------------------------------------------------------
assert _assign['peer_group'].notna().all()
assert (_assign['peer_group'] != '').all()
assert (_assign['peer_group'] != 'unassigned').all()

_peer_dist = pd.read_parquet(_SHARED_ROOT / 'peer_distributions.parquet')
_pd_groups = set(_peer_dist['regime_bucket'])
_as_groups = set(_assign['peer_group'])
assert _as_groups == _pd_groups, (
    f"❌ Peer groups in assignments ({len(_as_groups)}) differ from "
    f"distributions ({len(_pd_groups)})"
)
print(f"✅ All {len(_as_groups)} peer groups have a distribution")

# ---------------------------------------------------------------------------
# 15d  Peer distribution sanity
# ---------------------------------------------------------------------------
for _col in ['p10_days', 'p50_days', 'p90_days']:
    assert _peer_dist[_col].notna().all(), f"❌ NaN in {_col}"
    assert (_peer_dist[_col] > 0).all(), f"❌ Non-positive {_col}"

assert (_peer_dist['p10_days'] <= _peer_dist['p50_days']).all()
assert (_peer_dist['p50_days'] <= _peer_dist['p90_days']).all()
print(f"✅ Peer distributions monotonic and positive")

# ---------------------------------------------------------------------------
# 15e  Risk-factor coverage
# ---------------------------------------------------------------------------
with open(_SHARED_ROOT / 'peer_risk_factors.json') as f:
    _risk = json.load(f)

_reliable_groups = set(_peer_dist[_peer_dist['reliable']]['regime_bucket'])
_scored_groups   = set(_risk['risk_factors'].keys())

_missing = _reliable_groups - _scored_groups
if _missing:
    raise RuntimeError(
        f"❌ {len(_missing)} reliable peer group(s) lack SHAP attributions: "
        f"{list(_missing)[:5]}"
    )
print(f"✅ All reliable peer groups have SHAP attributions "
      f"({len(_scored_groups)} of {len(_reliable_groups)} reliable)")

# ---------------------------------------------------------------------------
# 15f  Feature pool covers the audit features
# ---------------------------------------------------------------------------
with open(_SHARED_ROOT / 'audit_feature_pool.json') as f:
    _pool = json.load(f)
_missing = [f for f in _pool['features'] if f not in df.columns]
assert not _missing, f"❌ Audit features missing from df: {_missing[:10]}"
print(f"✅ audit_feature_pool.json: {_pool['n_features']} features, all present")

# ---------------------------------------------------------------------------
# 15g  Eta² gate honesty (still relevant — peer axis must separate)
# ---------------------------------------------------------------------------
with open(_SHARED_ROOT / 'strategy_manifest.json') as f:
    _manifest = json.load(f)
print(f"✅ strategy_manifest.json references {_manifest['n_peer_groups']} peer groups")

print("\n" + "=" * 70)
print("✅ PIPELINE VALIDATION PASSED")
print("=" * 70)


PIPELINE VALIDATION — PEER-GROUP CONTRACT
✅ cluster_data\shared\trial_reference_corpus.parquet
✅ cluster_data\shared\canonical_split.parquet
✅ cluster_data\shared\peer_group_assignments.parquet
✅ cluster_data\shared\peer_distributions.parquet
✅ cluster_data\shared\peer_feature_profiles.json
✅ cluster_data\shared\peer_risk_factors.json
✅ cluster_data\shared\audit_feature_pool.json
✅ cluster_data\shared\text_feature_inventory.json
✅ cluster_data\shared\temporal_reference.json
✅ cluster_data\shared\eligibility_text_features.parquet
✅ cluster_data\shared\strategy_manifest.json

✅ Trial universe consistent across all artifacts: 171,881
✅ All 44 peer groups have a distribution
✅ Peer distributions monotonic and positive
✅ All reliable peer groups have SHAP attributions (39 of 39 reliable)
✅ audit_feature_pool.json: 107 features, all present
✅ strategy_manifest.json references 44 peer groups

✅ PIPELINE VALIDATION PASSED


In [28]:
# ============================================================================
# CELL 16: HANDOFF BUNDLE FOR AUDIT NOTEBOOK
# ============================================================================
# Extended to pass model_r2_test and model_r2_gap through the handoff so
# that Notebook 03 can display both without re-reading peer_risk_factors.
# ============================================================================

from pathlib import Path
import json
from datetime import datetime, timezone

_SHARED_ROOT = Path('cluster_data/shared')

print("\n" + "=" * 78)
print("HANDOFF BUNDLE FOR AUDIT NOTEBOOK")
print("=" * 78)

with open(_SHARED_ROOT / 'strategy_manifest.json') as f:
    _manifest = json.load(f)
with open(_SHARED_ROOT / 'text_feature_inventory.json') as f:
    _text_inv = json.load(f)
with open(_SHARED_ROOT / 'temporal_reference.json') as f:
    _temporal = json.load(f)
with open(_SHARED_ROOT / 'audit_feature_pool.json') as f:
    _audit_pool = json.load(f)
with open(_SHARED_ROOT / 'peer_risk_factors.json') as f:
    _risk = json.load(f)

# FIX (Priority #2): expose both R² values. Schema_version bumped to 2.
_handoff = {
    'schema_version':  2,   # bumped: reference_metrics now includes test R²
    'generated_by':    '02_clinical_trial_peer_grouping',
    'generated_at':    datetime.now(timezone.utc).isoformat(),

    'mode':            'peer_group_benchmarking_audit',

    'peer_axis': {
        'method':         'deterministic_branch_specific_slice',
        'interventional': 'stage × scope',
        'observational':  'obs::model::perspective',
        'n_peer_groups':  _manifest['n_peer_groups'],
    },

    'inputs': {
        'trial_id_col':           _manifest['trial_id_col'],
        'reference_corpus_path':  _manifest['paths']['reference_corpus'],
        'canonical_split_path':   _manifest['paths']['canonical_split'],
        'peer_assignments_path':  _manifest['paths']['peer_assignments'],
        'peer_distributions_path':_manifest['paths']['peer_distributions'],
        'peer_feature_profiles_path': _manifest['paths']['peer_feature_profiles'],
        'peer_risk_factors_path': _manifest['paths']['peer_risk_factors'],
        'audit_feature_pool_path':_manifest['paths']['audit_feature_pool'],
        'text_inventory_path':    _manifest['paths']['text_feature_inventory'],
        'temporal_reference_path':_manifest['paths']['temporal_reference'],
        'eligibility_text_path':  _manifest['paths']['eligibility_text'],

        'audit_feature_pool':     _audit_pool['features'],
        'text_feature_inventory': _text_inv,
        'temporal_reference':     _temporal,
    },

    'reference_metrics': {
        'n_trials':             _manifest['n_trials'],
        'n_peer_groups':        _manifest['n_peer_groups'],
        'n_peer_groups_reliable': int(
            pd.read_parquet(_SHARED_ROOT / 'peer_distributions.parquet')['reliable'].sum()
        ),
        'attribution_model':    _risk['model_family'],
        'attribution_r2_train': _risk['model_r2_train'],
        # FIX (Priority #2): R² on the held-out test fold, plus the gap.
        # If these are missing (e.g. an older peer_risk_factors.json), the
        # audit notebook must tolerate their absence.
        'attribution_r2_test':  _risk.get('model_r2_test'),
        'attribution_r2_gap':   _risk.get('model_r2_gap'),
    },

    'instructions_for_03': {
        'primary': (
            "Given a finished trial's design features and its ACTUAL duration, "
            "report where the duration sits within its peer-group distribution "
            "and attribute the position to design features."
        ),
        'steps': [
            "1. Assign the trial to a peer group using build_strategy_A_partition.",
            "2. Look up the peer group in peer_distributions.parquet.",
            "3. Compute the trial's percentile within that peer group.",
            "4. Flag whether the trial is inside or outside the peer's p05–p95 range.",
            "5. Attribute the position using peer_risk_factors.json for the "
            "   assigned peer group.",
            "6. Emit an audit report.",
        ],
        'must_not': [
            "Fit any model on the test fold.",
            "Use a routing classifier — peer assignment is deterministic.",
            "Report a predicted duration — the audit is retrospective.",
        ],
    },

    'no_deployment_decision': True,
    'notes': (
        "This handoff does not contain a deployment verdict. There is no "
        "routed system and no global-vs-routed comparison. The audit's job "
        "is to benchmark a finished trial against its peers."
    ),
}

_handoff_path = Path('cluster_data/handoff_to_03.json')
with open(_handoff_path, 'w') as f:
    json.dump(_handoff, f, indent=2, default=str)

print(f"\n✅ Wrote {_handoff_path}")
print(f"   Mode:                  {_handoff['mode']}")
print(f"   Peer groups:           {_handoff['peer_axis']['n_peer_groups']}")
print(f"   Reference trials:      {_handoff['reference_metrics']['n_trials']:,}")
print(f"   Audit feature pool:    {len(_handoff['inputs']['audit_feature_pool'])}")
print(f"   Attribution R² (train): {_handoff['reference_metrics']['attribution_r2_train']:.4f}")
if _handoff['reference_metrics']['attribution_r2_test'] is not None:
    print(f"   Attribution R² (test):  {_handoff['reference_metrics']['attribution_r2_test']:.4f}")
    print(f"   Attribution R² gap:     {_handoff['reference_metrics']['attribution_r2_gap']:+.4f}")
print(f"\n📌 03 reads this file to build the audit report.")
print("=" * 78)


HANDOFF BUNDLE FOR AUDIT NOTEBOOK

✅ Wrote cluster_data\handoff_to_03.json
   Mode:                  peer_group_benchmarking_audit
   Peer groups:           44
   Reference trials:      171,881
   Audit feature pool:    107
   Attribution R² (train): 0.4134
   Attribution R² (test):  0.3819
   Attribution R² gap:     +0.0316

📌 03 reads this file to build the audit report.
